# DOH/TRAMS

## Clean, combine and prepare data

In [ ]:
import pandas as pd
import glob, os, re, csv

def _sniff_delimiter(fp, encodings=('utf-8','cp874','latin-1')):
    """
    Open the first line of `fp` using each encoding until we can
    sniff its delimiter with csv.Sniffer.
    """
    for enc in encodings:
        try:
            with open(fp, encoding=enc) as f:
                sample = f.readline()
            dialect = csv.Sniffer().sniff(sample)
            return dialect.delimiter, enc
        except (UnicodeDecodeError, csv.Error):
            continue
    raise ValueError(f"Could not sniff delimiter/encoding for {fp}")

def _read_file(fp):
    """
    Sniff delimiter & encoding, then pd.read_csv with those.
    """
    sep, enc = _sniff_delimiter(fp)
    return pd.read_csv(fp, sep=sep, encoding=enc)

def combine_accident_years(folder_path, pattern="accident*.csv", year_col="ปีที่เกิดเหตุ"):
    """
    Reads accident2019.csv…accident2025.csv from `folder_path`, sniffing each file’s
    delimiter & encoding, extracts your specified year_col (which holds YYYY),
    filters to 2019-2025, and concatenates.
    """
    files = glob.glob(os.path.join(folder_path, pattern))
    dfs = []

    for fp in files:
        # skip out-of-range files by filename
        m = re.search(r"accident(\d{4})\.csv$", os.path.basename(fp), re.IGNORECASE)
        if not m:
            continue
        yr = int(m.group(1))
        if yr < 2019 or yr > 2024:
            continue

        df = _read_file(fp)
        print(df.columns.tolist())

        if year_col not in df.columns:
            raise KeyError(f"Column '{year_col}' not in {os.path.basename(fp)}; got columns: {df.columns.tolist()}")

        # year_col already contains YYYY, so cast
        df['year'] = df[year_col].astype(int)
        dfs.append(df)

    if not dfs:
        raise RuntimeError("No valid files loaded-check your path & filename pattern.")

    combined = pd.concat(dfs, ignore_index=True)
    # ensure only 2019-2025
    return combined[(combined['year'] >= 2019) & (combined['year'] <= 2025)]

# === USAGE ===
folder = "../data/DOH"
df_all = combine_accident_years(folder_path=folder)
print(f"Loaded {len(df_all)} records spanning years {sorted(df_all['year'].unique())}")
print(df_all.head())
# df_all.to_csv(os.path.join(folder, "accidents_2019-2025.csv"), index=False)

# for yr in sorted(df_all['year'].unique()):
#     ncols = df_all[df_all['year'] == yr].shape[1]
#     print(f"Year {yr}: {ncols} columns")

In [ ]:
for yr in sorted(df_all['year'].unique()):
    ncols = df_all[df_all['year'] == yr].shape[1]
    print(f"Year {yr}: {ncols} columns")

# Mapping of Thai column names to English
col_rename = {
    'ปีที่เกิดเหตุ': 'Year',
    'วันที่เกิดเหตุ': 'AccidentDate',
    'เวลา': 'AccidentTime',
    'วันที่รายงาน': 'ReportDate',
    'เวลาที่รายงาน': 'ReportTime',
    'ACC_CODE': 'AccidentCode',
    'หน่วยงาน': 'Agency',
    'สายทางหน่วยงาน': 'AgencyRoute',
    'รหัสสายทาง': 'RouteCode',
    'สายทาง': 'Route',
    'KM': 'KM',
    'จังหวัด': 'Province',
    'รถคันที่1': 'FirstVehicle',
    'บริเวณที่เกิดเหตุ': 'AccidentLocation',
    'มูลเหตุสันนิษฐาน': 'PresumedCause',
    'ลักษณะการเกิดเหตุ': 'AccidentType',
    'สภาพอากาศ': 'Weather',
    'LATITUDE': 'Latitude',
    'LONGITUDE': 'Longitude',
    'รถที่เกิดเหตุ': 'VehiclesInvolved',
    'รถและคนที่เกิดเหตุ': 'VehiclesAndPeopleInvolved',
    'รถจักรยานยนต์': 'Motorcycle',
    'รถสามล้อเครื่อง': 'MotorTricycle',
    'รถยนต์นั่งส่วนบุคคล': 'PrivateCar',
    'รถตู้': 'Van',
    'รถปิคอัพโดยสาร': 'PickupPassenger',
    'รถโดยสารมากกว่า4ล้อ': 'BusOver4Wheels',
    'รถปิคอัพบรรทุก4ล้อ': 'PickupTruck4Wheels',
    'รถบรรทุก6ล้อ': 'Truck6Wheels',
    'รถบรรทุกไม่เกิน10ล้อ': 'TruckUpTo10Wheels',
    'รถบรรทุกมากกว่า10ล้อ': 'TruckOver10Wheels',
    'รถอีแต๋น': 'E-TanTruck',
    'รถอื่นๆ': 'OtherVehicles',
    'คนเดินเท้า': 'Pedestrian',
    'ผู้เสียชีวิต': 'Fatalities',
    'ผู้บาดเจ็บสาหัส': 'SeriousInjuries',
    'ผู้บาดเจ็บเล็กน้อย': 'MinorInjuries',
    'รวมจำนวนผู้บาดเจ็บ': 'TotalInjuries',
    'year': 'YearNum'
}

# To rename columns in your DataFrame:
df_all = df_all.rename(columns=col_rename)
df_all = df_all.drop(columns=['YearNum'])


### Harmonize 2024 date and time fields

In [ ]:
# Fix 2024's date/time columns to match other years
mask_2024 = df_all['Year'] == 2024

# Convert Excel serial date to string date for AccidentDate and ReportDate
def excel_serial_to_date(serial):
    return pd.to_datetime('1899-12-30') + pd.to_timedelta(serial, 'D')

df_all.loc[mask_2024, 'AccidentDate'] = df_all.loc[mask_2024, 'AccidentDate'].apply(excel_serial_to_date).dt.strftime('%-m/%-d/%Y')
df_all.loc[mask_2024, 'ReportDate'] = df_all.loc[mask_2024, 'ReportDate'].apply(excel_serial_to_date).dt.strftime('%-m/%-d/%Y')

# Convert Excel time fraction to string time for AccidentTime and ReportTime
def excel_time_to_str(timeval):
    # Handles both float and string input
    try:
        t = float(timeval)
        hours = int(t * 24)
        minutes = int((t * 24 * 60) % 60)
        return f"{hours}:{minutes:02d}"
    except Exception:
        return timeval

df_all.loc[mask_2024, 'AccidentTime'] = df_all.loc[mask_2024, 'AccidentTime'].apply(excel_time_to_str)
df_all.loc[mask_2024, 'ReportTime'] = df_all.loc[mask_2024, 'ReportTime'].apply(excel_time_to_str)

In [ ]:
# def fraction_to_time_str(fraction):
#     if pd.isnull(fraction):
#         return None
#     total_seconds = int(round(fraction * 86400))
#     hours = total_seconds // 3600
#     minutes = (total_seconds % 3600) // 60
#     seconds = total_seconds % 60
#     return f"{hours:02d}:{minutes:02d}:{seconds:02d}"

# mask = df_all['Year'] == 2024

# # Convert ReportDate from an Excel serial date to datetime by first converting to numeric
# df_all.loc[mask, 'ReportDate'] = pd.to_datetime(
#     pd.to_numeric(df_all.loc[mask, 'ReportDate'], errors='coerce'), unit='D', origin='1899-12-30'
# )

# # Clean AccidentTime and ReportTime columns converting fractions of a day to time strings
# df_all.loc[mask, 'AccidentTime'] = df_all.loc[mask, 'AccidentTime'].apply(fraction_to_time_str)
# df_all.loc[mask, 'ReportTime'] = df_all.loc[mask, 'ReportTime'].apply(fraction_to_time_str)

# # Optionally, fill missing AccidentDate with ReportDate
# df_all.loc[mask, 'AccidentDate'] = df_all.loc[mask, 'AccidentDate'].fillna(df_all.loc[mask, 'ReportDate'])

# print(df_all[df_all['Year'] == 2024].head())

In [ ]:
df_all.to_csv(os.path.join("../data/", "TRAMS_2019-2024.csv"), index=False)

## Exploratory DOH

In [ ]:
import os
import calendar
from pathlib import Path

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
from shapely.geometry import box
import numpy as np
import re

import osmnx


In [ ]:
df_all = pd.read_csv(os.path.join("TRAMS_2019-2024.csv"))
print(df_all.head())
print(f"Final DataFrame shape: {df_all.shape}")
print(f"Columns: {df_all.columns.tolist()}")

In [ ]:
df_all.info()
print(df_all.describe(include='all'))

In [ ]:

print('Number of Incidents 2019-2024: ', df_all.shape[0])
print('__________________________________________________________________')

print('Serious Injuries 2019-2024: ', df_all['SeriousInjuries'].sum())
print('Minor Injuries 2019-2024: ', df_all['MinorInjuries'].sum())
print('Injuries 2019-2024: ',df_all['TotalInjuries'].sum())
print('Fatalities 2019-2024: ', df_all['Fatalities'].sum())

print('Casualties (total injuries + fatalities) 2019-2024: ', df_all['TotalInjuries'].sum() + df_all['Fatalities'].sum() )

print('__________________________________________________________________')
print('Vehicles Involved 2019-2024: ', df_all['VehiclesInvolved'].sum())
print('Vehicles And People Involved 2019-2024: ', df_all['VehiclesAndPeopleInvolved'].sum())

print('__________________________________________________________________')
print('Number of fatalities per year:',df_all.groupby('Year')['Fatalities'].sum())
# print(df_all.groupby('Year')['Fatalities'].sum())

Recorded yearly row counts from the source files:

- 20,015
- 17,744
- 17,573
- 17,347
- 17,735
- 17,209

### Missingness

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- Step 1: Calculate missing percentage per column per year ---
# Group by 'Year', compute fraction of missing values for each column, convert to percentage
missing_by_year = df_all.groupby('Year').apply(lambda g: g.isnull().mean() * 100).T

# --- Step 2: Sort columns (variables) by average missingness ---
missing_by_year["Mean"] = missing_by_year.mean(axis=1)
missing_by_year = missing_by_year.sort_values(by="Mean", ascending=False)
missing_by_year = missing_by_year.drop(columns="Mean")

# --- Step 3: Plot heatmap ---
plt.figure(figsize=(10, 12))
ax = sns.heatmap(
    missing_by_year,
    cmap="Reds",
    annot=True,
    fmt=".1f",
    annot_kws={"size": 13},
    cbar_kws={"shrink": 0.8, "pad": 0.02}
)

ax.set_xticklabels(ax.get_xticklabels(), fontsize=11, rotation=0)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=11, rotation=0)

plt.title("Heatmap of Missing Values (%) by Variable and Year", fontsize=20)
plt.xlabel("Year", fontsize=14)
plt.ylabel("Variables", fontsize=14)
plt.tight_layout()
plt.tight_layout()
plt.savefig("fig/heatmap_all_missing.pdf", dpi=300)
plt.show()

# ../data/DOH/missing_values_heatmap.png


# Filter motorcycle involved

In [ ]:
df_motorcycle = df_all[df_all['Motorcycle'] > 0]
df_motorcycle.info()

In [ ]:
# Summary statistics for motorcycle-involved accidents
print('--- Motorcycle-Involved Incident Statistics (2019-2024) ---')
print('Number of Motorcycle Incidents:', df_motorcycle.shape[0])
print('-----------------------------------------------------------')

# Injury and fatality counts
print('Fatalities:', df_motorcycle['Fatalities'].sum())
print('Serious Injuries:', df_motorcycle['SeriousInjuries'].sum())
print('Minor Injuries:', df_motorcycle['MinorInjuries'].sum())
print('Total Injuries:', df_motorcycle['TotalInjuries'].sum())
print('Total Casualties (injuries + fatalities):', df_motorcycle['TotalInjuries'].sum() + df_motorcycle['Fatalities'].sum())
print('-----------------------------------------------------------')

# Vehicle involvement
print('Total Vehicles Involved:', df_motorcycle['VehiclesInvolved'].sum())
print('Total Vehicles And People Involved:', df_motorcycle['VehiclesAndPeopleInvolved'].sum())

# Calculate percentages compared to all accidents
motorcycle_fatality_rate = df_motorcycle['Fatalities'].sum() / df_motorcycle['VehiclesAndPeopleInvolved'].sum() * 100
print('-----------------------------------------------------------')
print(f'Fatality rate per involved person: {motorcycle_fatality_rate:.2f}%')

# Distribution by year
yearly_counts = df_motorcycle.groupby('Year').size()
print('-----------------------------------------------------------')
print('Motorcycle Accidents by Year:')
print(yearly_counts)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# --- Step 1: Calculate missing percentage per column per year ---
# Group by 'Year', compute fraction of missing values for each column, convert to percentage
missing_by_year_motorcycle = df_motorcycle.groupby('Year').apply(lambda g: g.isnull().mean() * 100).T

# --- Step 2: Sort columns (variables) by average missingness ---
missing_by_year_motorcycle["Mean"] = missing_by_year_motorcycle.mean(axis=1)
missing_by_year_motorcycle = missing_by_year_motorcycle.sort_values(by="Mean", ascending=False)
missing_by_year_motorcycle = missing_by_year_motorcycle.drop(columns="Mean")

# --- Step 3: Plot heatmap ---
plt.figure(figsize=(10, 12))
ax = sns.heatmap(
    missing_by_year_motorcycle,
    cmap="Reds",
    annot=True,
    fmt=".1f",
    annot_kws={"size": 13},
    cbar_kws={"shrink": 0.8, "pad": 0.02}
)

ax.set_xticklabels(ax.get_xticklabels(), fontsize=11, rotation=0)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=11, rotation=0)

plt.title("Heatmap of Missing Values (%) by Variable and Year", fontsize=20)
plt.xlabel("Year", fontsize=14)
plt.ylabel("Variables", fontsize=14)
plt.tight_layout()
plt.tight_layout()
# plt.savefig("fig/heatmap_motorcycle_missing.pdf", dpi=300)
plt.show()


# Translate Thai to English

In [ ]:
# print unique values in AccidentLocation
print("Unique values in 'AccidentLocation':", df_motorcycle['AccidentLocation'].unique())
# how many unique values in AccidentLocation
print("Number of unique values in 'AccidentLocation':", df_motorcycle['AccidentLocation'].nunique())

print("-" * 50)
# PresumedCause
print("Unique values in 'PresumedCause':", df_motorcycle['PresumedCause'].unique())
# how many unique values in PresumedCause
print("Number of unique values in 'PresumedCause':", df_motorcycle['PresumedCause'].nunique())
print("-" * 50)

# AccidentType
print("Unique values in 'AccidentType':", df_motorcycle['AccidentType'].unique())
# how many unique values in AccidentType
print("Number of unique values in 'AccidentType':", df_motorcycle['AccidentType'].nunique())
print("-" * 50)

# Weather
print("Unique values in 'Weather':", df_motorcycle['Weather'].unique())
# how many unique values in Weather
print("Number of unique values in 'Weather':", df_motorcycle['Weather'].nunique())
print("-" * 50)

# FirstVehicle
print("Unique values in 'FirstVehicle':", df_motorcycle['FirstVehicle'].unique())
# how many unique values in FirstVehicle
print("Number of unique values in 'FirstVehicle':", df_motorcycle['FirstVehicle'].nunique())


### Translate FirstVehicle

In [ ]:
translation_dict = {
    'รถจักรยานยนต์': 'Motorcycle',
    'รถปิคอัพบรรทุก 4 ล้อ': 'PickupTruck4Wheels',
    'คนเดินเท้า': 'Pedestrian',
    'รถสามล้อเครื่อง': 'MotorTricycle',
    'รถยนต์นั่งส่วนบุคคล/รถยนต์นั่งสาธารณะ': 'PrivateCar',
    'รถตู้': 'Van',
    'รถจักรยาน': 'Bicycle',
    'รถโดยสารขนาดใหญ่': 'BusOver4Wheels',
    'รถบรรทุกมากกว่า 6 ล้อ ไม่เกิน 10 ล้อ': 'TruckUpTo10Wheels',
    'รถบรรทุกมากกว่า 10 ล้อ (รถพ่วง)': 'TruckOver10Wheels',
    'รถปิคอัพโดยสาร': 'PickupPassenger',
    'รถบรรทุก 6 ล้อ': 'Truck6Wheels',
    'รถอีแต๋น/เพื่อการเกษตร': 'E-TanTruck',
    'อื่นๆ': 'OtherVehicles',
    'รถสามล้อ': 'Tricycle'
}

df_motorcycle['FirstVehicle'] = df_motorcycle['FirstVehicle'].replace(translation_dict)


### Translate AccidentLocation

In [ ]:
accident_location_dict = {
    'ทางตรง+ไม่มีความลาดชัน': 'Straight, Flat',
    'ทางสี่แยก': 'Four-way Intersection',
    'ทางโค้งกว้าง+ที่ลาดชัน': 'Wide Curve, Slope',
    'ทางโค้งกว้าง+ไม่มีความลาดชัน': 'Wide Curve, No Slope',
    'ทางสามแยก (T)': 'T-Intersection',
    'ทางเชื่อมเข้าพื้นที่สาธารณะ/เชิงพาณิชย์': 'Access to Public/Commercial Area',
    'ทางโค้งหักศอก+ที่ลาดชัน': 'Sharp Curve, Slope',
    'ทางตรง+ที่ลาดชัน': 'Straight, Slope',
    'ทางโค้งหักศอก+ไม่มีความลาดชัน': 'Sharp Curve, No Slope',
    'ทางเชื่อมเข้าพื้นที่ส่วนบุคคล': 'Access to Private Area',
    'ทางเชื่อมเข้าบริเวณหน้าโรงเรียน': 'Access to School Front',
    'ทางสามแยก (Y)': 'Y-Intersection',
    'วงเวียน': 'Roundabout',
    'จุดกลับรถต่างระดับ': 'Multi-level Turnaround',
    'ทางตรง': 'Straight Road',
    'ทางโค้งกว้าง': 'Wide Curve',
    'ทางแยกต่างระดับ/Ramps': 'Grade-separated/Ramps',
    'สะพาน (สะพานข้ามแม่น้ำ/ข้ามคลอง)': 'Bridge',
    'ทางโค้งหักศอก': 'Sharp Bend',
    'ทางสามแยก(T)': 'T-Intersection',
    'ทางตรง+ไม่มีความลาดชัน/ที่ราบ': 'Straight, Flat',
    'ทางตรง+ที่ราบ': 'Straight, Flat',
    'เข้าพื้นที่ส่วนบุคคล': 'Entering Private Area',
    'เข้าสถานศึกษา / สถานที่ราชการ': 'Access to School/Government Area',
    'ทางแยกรูปตัว T+ที่ราบ': 'T-Intersection, Flat',
    'ทางโค้งหักศอก+ก้นโค้ง': 'Sharp Bend, Bottom of Curve',
    'ทางแยกต่างระดับ / Ramps+ที่ราบ': 'Grade-separated/Ramps , Flat',
    'ทางโค้งปกติ+ที่ราบ': 'Normal Curve,Flat',
    'ทางตรง+โค้งดิ่ง/ที่ลาดชัน': 'Straight with Steep Slope',
    'เข้าพื้นที่สาธารณะหรือเชิงพานิชย์': 'Entering Public/Commercial Area',
    'ทางโค้งกว้าง+ไม่มีความลาดชัน/ที่ราบ': 'Wide Curve,Flat',
    'ทางแยกรูปตัว ++ที่ราบ': 'Complex Intersection,Flat',
    'ทางคนเดินเท้า+ที่ราบ': 'Pedestrian Path,Flat',
    'ทางโค้งกว้าง+โค้งดิ่ง/ที่ลาดชัน': 'Wide Curve with Steep Slope',
    'ทางโค้งปกติ+ก้นโค้ง': 'Normal Curve at Bottom',
    'ทางตรง+บนช่วงลาดชัน': 'Straight,Slope',
    'ทางโค้งหักศอก+ไม่มีความลาดชัน/ที่ราบ': 'Sharp Curve,Flat',
    'ทางลอด+ที่ราบ': 'Underpass,Flat',
    'สะพาน+ยอดโค้ง': 'Bridge , Apex',
    'ทางโค้งหักศอก+ที่ราบ': 'Sharp Bend,Flat',
    'ทางแยกรูปตัว Y+ที่ราบ': 'Y-Intersection,Flat',
    'ทางโค้งปกติ+บนช่วงลาดชัน': 'Normal Curve,Slope',
    'ทางสามแยก(Y)': 'Y-Intersection',
    'จุดกลับรถต่างระดับ+ยอดโค้ง': 'Multi-level Turnaround , Apex',
    'สะพาน+ที่ราบ': 'Bridge,Flat',
    'สะพาน': 'Bridge',
    'ทางโค้งปกติ+ยอดโค้ง': 'Normal Curve , Apex',
    'จุดกลับรถต่างระดับ+ที่ราบ': 'Multi-level Turnaround,Flat',
    'ทางตรง+ยอดโค้ง': 'Straight Through Apex',
    'สะพาน+บนช่วงลาดชัน': 'Bridge,Slope',
    'วงเวียน+ที่ราบ': 'Roundabout,Flat',
    'ทางแยกรูปตัว Y+บนช่วงลาดชัน': 'Y-Intersection,Slope',
    'ทางม้าลาย+ที่ราบ': 'Crosswalk,Flat',
    'ทางโค้งหักศอก+โค้งดิ่ง/ที่ลาดชัน': 'Sharp Bend with Steep Curve',
    'ทางแยกต่างระดับ / Ramps+ยอดโค้ง': 'Grade-separated/Ramps , Apex',
    'ทางรถไฟตัดผ่าน+ที่ราบ': 'Railway Crossing,Flat',
    'ทางยกระดับ+ที่ราบ': 'Elevated Road,Flat',
    'ทางคนเดินเท้า': 'Pedestrian Path',
    'ทางแยกรูปตัว ++บนช่วงลาดชัน': 'T-Intersection,Slope',
    'ทางแยกรูปตัว T+บนช่วงลาดชัน': 'T-Intersection,Slope',
    'ทางแยกรูปตัว T+ก้นโค้ง': 'T-Intersection , Curve Bottom',
    'ทางแยกรูปตัว ++ยอดโค้ง': 'T-Intersection , Apex',
    'ทางแยกรูปตัว T+ยอดโค้ง': 'T-Intersection , Apex',
    'สถานี/จุดขึ้นลงระบบขนส่งสาธารณะ+ที่ราบ': 'Transit Station/Stop,Flat',
    'ทางลอด+บนช่วงลาดชัน': 'Underpass,Slope Section',
    'ทางตรง+ก้นโค้ง': 'Straight , Curve Bottom',
    'ทางโค้งหักศอก+บนช่วงลาดชัน': 'Sharp Bend,Slope',
    'มีการเปลี่ยนจำนวนช่องจราจร': 'Change in Number of Lanes',
    'จุดกลับรถต่างระดับ+ก้นโค้ง': 'Multi-level Turnaround , Curve Bottom',
    'มีการเปลี่ยนความกว้างช่องจราจร+ที่ราบ': 'Change in Lane Width,Flat',
    'มีการเปลี่ยนความกว้างช่องจราจร': 'Change in Lane Width',
    'ทางรถจักรยานยนต์+ที่ราบ': 'Motorcycle Lane,Flat',
    'ทางรถจักรยานยนต์': 'Motorcycle Lane',
    'ทางม้าลาย/คนเดินข้าม': 'Crosswalk',
}

df_motorcycle['AccidentLocation'] = df_motorcycle['AccidentLocation'].replace(accident_location_dict)


In [ ]:
import re
import numpy as np
import pandas as pd

# If you've already overwritten df_all['AccidentLocation'] with English,
# this uses it directly. Otherwise, do the replace first into a new column.
# df_all['AccidentLocation'] = df_all['AccidentLocation'].replace(accident_location_dict)

# --- Define canonical tags via regex on the ENGLISH phrases ---
# Order = output order in the tag string
tag_patterns = [
    ("STRAIGHT",               r"\bstraight( road| through apex)?\b"),
    ("CURVE_WIDE",             r"\bwide curve\b"),
    ("CURVE_SHARP",            r"\bsharp (curve|bend)\b"),
    ("CURVE_NORMAL",           r"\bnormal curve\b"),

    ("INTERSECTION_4WAY",      r"\bfour[- ]?way intersection\b"),
    ("INTERSECTION_T",         r"\bt-?intersection\b"),
    ("INTERSECTION_Y",         r"\by-?intersection\b"),
    ("INTERSECTION_COMPLEX",   r"\bcomplex intersection\b"),
    ("ROUNDABOUT",             r"\broundabout\b"),
    ("RAMPS",                  r"grade[- ]?separated|ramps?"),
    ("TURNAROUND_MULTILEVEL",  r"\bmulti[- ]?level turnaround\b"),
    ("BRIDGE",                 r"\bbridge\b"),
    ("UNDERPASS",              r"\bunderpass\b"),
    ("ELEVATED",               r"\belevated road\b"),
    ("RAILWAY_CROSSING",       r"\brailway crossing\b"),
    ("PEDESTRIAN_PATH",        r"\bpedestrian path\b"),
    ("CROSSWALK",              r"\bcrosswalk\b"),
    ("MOTORCYCLE_LANE",        r"\bmotorcycle lane\b"),
    ("TRANSIT_STOP",           r"\btransit station/stop\b|\bstation/stop\b"),

    ("ACCESS_PUBLIC_COMMERCIAL", r"access to public/commercial area|entering public/commercial area"),
    ("ACCESS_PRIVATE",           r"access to private area|entering private area"),
    ("ACCESS_SCHOOL_GOV",        r"access to school front|access to school/government area"),

    ("FLAT",                   r"\bflat\b|no slope|level"),
    ("SLOPE_STEEP",            r"steep slope|steep curve|with steep slope|with steep curve"),
    ("SLOPE",                  r"\bslope\b|slope section"),
    ("APEX",                   r"\bapex\b"),
    ("BOTTOM",                 r"\bbottom of curve\b|\bcurve bottom\b"),

    ("LANE_COUNT_CHANGE",      r"change in number of lanes"),
    ("LANE_WIDTH_CHANGE",      r"change in lane width"),
]

def loc_to_tags(s: str) -> str | float:
    if not isinstance(s, str):
        return np.nan
    txt = re.sub(r"\s+", " ", s.strip().lower())
    found = set()
    for tag, pat in tag_patterns:
        if re.search(pat, txt, flags=re.IGNORECASE):
            found.add(tag)
    if not found:
        return "OTHER"
    # stable order by tag_patterns
    ordered = [tag for tag, _ in tag_patterns if tag in found]
    return ",".join(ordered)

# Build comma-separated multi-label string
df_motorcycle["AccidentLocation_tags"] = df_motorcycle["AccidentLocation"].apply(loc_to_tags)




In [ ]:
# (Optional) one-hot columns per tag
# Split tags and get dummies
# Note: keep_na=False so NaN rows won't create "nan" tag
ml = df_all["AccidentLocation_tags"].str.get_dummies(sep=",")
# Ensure all defined tags exist as columns (even if absent)
for tag, _ in tag_patterns:
    if tag not in ml.columns:
        ml[tag] = 0
# Add to df_all (prefixed)
ml = ml[[tag for tag, _ in tag_patterns]]  # enforce order
ml.columns = [f"AL_{c}" for c in ml.columns]
df_all_with_location_tags = pd.concat([df_all, ml], axis=1)

### Translate AccidentCause Categories

- **SPEEDING**
- **IMPAIRMENT**  
    _Alcohol/drugs_
- **DISTRACTION**  
    _Phone, in-/external distractions_
- **DRIVER_CONDITION**  
    _Drowsy, fatigue, medical issues, vision problems_
- **MANOEUVRE**  
    _Overtake, lane-change, cut-in, sudden braking_
- **YIELD_SIGNAL**  
    _Failure to yield or signal; disobeying signs/lights_
- **LANE_DISCIPLINE**  
    _Wrong-way, wrong lane, straddling, not keeping left_
- **VISIBILITY_LIGHTING**  
    _Poor lighting, reduced visibility, occlusion_
- **ROAD_ENVIRONMENT**  
    _Road surface, narrow roads, geometry/curves, inadequate roadside clear zone_
- **TRAFFIC_CONTROL**  
    _Signal malfunction/absence; faulty signs, markings, or lighting_
- **VEHICLE_OPS_DEFECT**  
    _Defects in brakes, tires, engine, steering, electrical systems; overloading; slow vehicle performance_
- **ROADWORKS_OBSTRUCTION**  
    _Roadworks, objects/obstructions, disabled vehicle lacking warning_
- **EXPERIENCE**  
    _Unfamiliar route, left-side driving, inexperienced driving_
- **OTHER_UNKNOWN**


In [ ]:
presumed_cause_translation = {
    'แซงรถอย่างผิดกฎหมาย': 'Illegal Overtaking',
    'ขับรถย้อนศร': 'Wrong-Way Driving',
    'ขับรถเร็วเกินอัตรากำหนด': 'Speeding',
    'ขับรถเร็วเกินอัตราที่กำหนด': 'Speeding',
    'เมาสุรา': 'Drunk Driving',
    'แสงสว่างไม่เพียงพอ': 'Poor Lighting/Visibility',
    'แสงสว่างไม่เพียงพอหรือทัศนวิสัยกลางคืนไม่ดี': 'Poor Lighting/Visibility',
    'คน/รถ/สัตว์ตัดหน้ากระชั้นชิด': 'Sudden Cut-In Ahead',
    'มีการตัดหน้าระยะกระชั้นชิด': 'Sudden Cut-In Ahead',
    'ฝ่าฝืนสัญญาณไฟ/เครื่องหมายจราจร': 'Disobeying Signals/Signs',
    'ฝ่าฝืนสัญญาณไฟ / เครื่องหมายจราจร': 'Disobeying Signals/Signs',
    'ไม่คุ้นเคยเส้นทาง/ขับรถไม่ชำนาญ': 'Unfamiliar Route/Inexperienced',
    'ขับรถไม่ชำนาญ / ไม่เป็น': 'Inexperienced Driving',
    'หลับใน': 'Drowsy Driving (Asleep at Wheel)',
    'ขับรถตามกระชั้นชิด': 'Tailgating',
    'มีกองวัสดุ/สิ่งกีดขวาง': 'Obstruction on Roadway',
    'มีสิ่งกีดขวางบนทางหลวง (วัสดุหรือสัตว์)': 'Obstruction on Roadway',
    'มีสิ่งบดบังการมองเห็น': 'Obstructed View',
    'ทางโค้งอันตราย': 'Dangerous Curve',
    'หยุดรถกะทันหัน': 'Sudden Braking',
    'ฝ่าฝืนป้ายหยุดขณะออกจากทางร่วมทางแยก': 'Disobey Stop Sign (Junction Exit)',
    'ฝ่าฝืนป้ายหยุดขณะออกจากทางร่วมแยก': 'Disobey Stop Sign (Junction Exit)',
    'ระบบสัญญาณไฟจราจรขัดข้อง': 'Traffic Signal Malfunction',
    'ขับรถผิดช่องทาง': 'Wrong Lane',
    'ไม่ให้สัญญาณชะลอ/เลี้ยว': 'Failure to Signal Slow/Turn',
    'ไม่ให้สัญญาณชะลอ / เลี้ยว': 'Failure to Signal Slow/Turn',
    'ไม่ให้สัญญาณเข้าจอด/ออกจากที่จอด': 'Failure to Signal Park/Unpark',
    'ไม่ให้สัญญาณเข้าจอด หรือออกจากที่จอด': 'Failure to Signal Park/Unpark',
    'ไม่ยอมให้รถที่มีสิทธิ์ไปก่อน': 'Failure to Yield',
    'ไม่ให้สิทธิรถที่มาก่อนผ่านทาง เช่น ทางแยก': 'Failure to Yield',
    'อุปกรณ์ยานพาหนะบกพร่อง': 'Vehicle Equipment Defect',
    'อุปกรณ์ยานพาหนะบกพร่อง (ระบุ)': 'Vehicle Equipment Defect',
    'เปลี่ยนช่องทางกะทันหัน': 'Sudden Lane Change',
    'ถนนชำรุด': 'Damaged Road',
    'ผิวถนนสภาพแย่หรือชำรุด': 'Damaged/Poor Road Surface',
    'ป้ายจราจรชำรุด': 'Damaged Traffic Sign',
    'เส้นแบ่งทิศทางจราจรชำรุด': 'Damaged Lane Markings',
    'ไม่มีเส้นแบ่งทิศทางจราจร': 'No Lane Markings',
    'ไม่มีป้ายจราจร': 'No Traffic Signs',
    'อื่นๆ': 'Other',
    'ใช้โทรศัพท์เคลื่อนที่ขณะขับรถ': 'Phone Use While Driving',
    'ใช้โทรศัพท์ขณะขับรถ': 'Phone Use While Driving',
    'ขับรถไม่เปิดไฟ/ไม่ใช้แสงสว่างตามกำหนด': 'No Required Lights',
    'ขับรถไม่เปิดไฟ / ไม่ใช้แสงสว่างตามกำหนด': 'No Required Lights',
    'รถเสียไม่แสดงเครื่องหมายหรือสัญญาณไฟที่กำหนด': 'Disabled Vehicle No Warning',
    'รถเสียไม่แสดงเครื่องหมาย/สัญญาณตามที่กำหนด': 'Disabled Vehicle No Warning',
    'ขับรถคร่อมเส้นแบ่งทิศทาง': 'Straddling Lane Line',
    'โรคประจำตัว': 'Medical Condition',
    'ถนนแคบ': 'Narrow Road',
    'ถนนลื่น': 'Slippery Road',
    'ถนนลื่น (เนื่องจากสภาพอากาศ)': 'Slippery Road',
    'ระบบไฟฟ้าของยานพาหนะขัดข้อง': 'Vehicle Electrical Fault',
    'การซ่อม/สร้างบนสายทาง': 'Road Works/Construction',
    'ยางเสื่อมสภาพ/ยางแตก': 'Tire Failure',
    'ระยะการมองเห็นไม่เพียงพอ': 'Insufficient Sight Distance',
    'ความเมื่อยล้า': 'Fatigue',
    'ไม่ให้สัญญาณเข้าจอด หรือออกจากที่จอด': 'Failure to Signal Park/Unpark',
    'ไฟส่องสว่างชำรุด': 'Road Lighting Fault',
    'ไฟฟ้าแสงสว่างชำรุด': 'Road Lighting Fault',
    'เบรคชำรุด': 'Brake Failure',
    'ระบบห้ามล้อขัดข้อง/ระบบเบรกชำรุด': 'Brake Failure',
    'มีคราบสะสมบนผิวถนน เช่น โคลน น้ำมัน': 'Slippery Contaminants on Road',
    'มีสิ่งรบกวนภายในรถ เช่น มีเด็ก มีสัตว์ภายในรถ': 'In-Vehicle Distraction',
    'มีสิ่งรบกวนภายในรถ': 'In-Vehicle Distraction',
    'มีสิ่งรบกวนภายนอกรถ': 'External Distraction',
    'ปัญหาทางสายตา': 'Vision Problems',
    'ไม่ขับรถในช่องทางเดินรถซ้ายสุดในถนนที่มี 4 ช่องทาง': 'Not Keeping Leftmost Lane (4-Lane)',
    'ยานพาหนะเคลื่อนที่ช้า': 'Slow-Moving Vehicle',
    'ระยะมองเห็นจำกัด': 'Limited Visibility',
    'ข้ามถนนโดยมีรถจอดหรือวัตถุข้างทางบดบังผู้ขับขี่': 'Crossing with Occlusion',
    'เครื่องยนต์ขัดข้อง': 'Engine Malfunction',
    'ลักษณะของถนน เช่น เป็นทางโค้ง, เนินเขา, ผิวทางแคบ': 'Road Geometry Factors',
    'การกระทำที่สุ่มเสี่ยงบนถนน': 'Risky Road Behaviour',
    'ไม่คุ้นเคยกับการขับขี่ด้านซ้าย เช่น กรณีชาวต่างชาติ': 'Unfamiliar with Left-Side Driving',
    'บรรทุกเกินอัตรา': 'Overloading',
    'ไม่มีระบบสัญญาณไฟจราจร': 'No Traffic Signal System',
    'ไม่มีมูลเหตุสันนิษฐานที่เกี่ยวข้องด้านสภาพสายทาง': 'No Road-Condition-Related Cause',
    'สูญเสียการควบคุม': 'Loss of Control',
    'ระบบบังคับเลี้ยวขัดข้อง': 'Steering System Fault',
    'ป้ายจราจรถูกบดบัง': 'Traffic Sign Obscured',
    'ระยะปลอดภัยข้างทางไม่เพียงพอ': 'Insufficient Roadside Clear Zone',
    'ขับขี่ช้าเกินเนื่องจากลักษณะของยานพาหนะ (เช่น แทรคเตอร์)': 'Slow-Moving Vehicle',
    'อาการป่วยหรือการไร้ความสามารถทางประสาทหรือร่างกาย': 'Medical Condition',
    'ความประมาท/เร่งรีบ (วิ่งตัดหน้า)': 'Reckless/Impatient Driving (Cutting In)',
    'ยางรถยนต์ชำรุด': 'Tire Failure',
    'ใช้สารออกฤทธิ์ต่อจิตและประสาท': 'Use of Psychoactive Substances'

}

# Apply
df_motorcycle['PresumedCause'] = df_motorcycle['PresumedCause'].replace(presumed_cause_translation)

In [ ]:
eng2cat = {
    'Speeding': 'SPEEDING',
    'Drunk Driving': 'IMPAIRMENT',
    'Use of Psychoactive Substances': 'IMPAIRMENT',
    'Phone Use While Driving': 'DISTRACTION',
    'In-Vehicle Distraction': 'DISTRACTION',
    'External Distraction': 'DISTRACTION',
    'Drowsy Driving (Asleep at Wheel)': 'DRIVER_CONDITION',
    'Fatigue': 'DRIVER_CONDITION',
    'Medical Condition': 'DRIVER_CONDITION',
    'Vision Problems': 'DRIVER_CONDITION',
    'Sudden Braking': 'MANOEUVRE',
    'Tailgating': 'MANOEUVRE',
    'Sudden Lane Change': 'MANOEUVRE',
    'Reckless/Impatient Driving (Cutting In)': 'MANOEUVRE',
    'Disobeying Signals/Signs': 'YIELD_SIGNAL',
    'Disobey Stop Sign (Junction Exit)': 'YIELD_SIGNAL',
    'Failure to Signal Slow/Turn': 'YIELD_SIGNAL',
    'Failure to Signal Park/Unpark': 'YIELD_SIGNAL',
    'Failure to Yield': 'YIELD_SIGNAL',
    'Wrong-Way Driving': 'LANE_DISCIPLINE',
    'Wrong Lane': 'LANE_DISCIPLINE',
    'Straddling Lane Line': 'LANE_DISCIPLINE',
    'Not Keeping Leftmost Lane (4-Lane)': 'LANE_DISCIPLINE',
    'Poor Lighting/Visibility': 'VISIBILITY_LIGHTING',
    'Limited Visibility': 'VISIBILITY_LIGHTING',
    'Insufficient Sight Distance': 'VISIBILITY_LIGHTING',
    'Obstructed View': 'VISIBILITY_LIGHTING',
    'Road Works/Construction': 'ROADWORKS_OBSTRUCTION',
    'Obstruction on Roadway': 'ROADWORKS_OBSTRUCTION',
    'Disabled Vehicle No Warning': 'ROADWORKS_OBSTRUCTION',
    'Traffic Signal Malfunction': 'TRAFFIC_CONTROL',
    'No Traffic Signal System': 'TRAFFIC_CONTROL',
    'Damaged Traffic Sign': 'TRAFFIC_CONTROL',
    'Traffic Sign Obscured': 'TRAFFIC_CONTROL',
    'Damaged Lane Markings': 'TRAFFIC_CONTROL',
    'No Lane Markings': 'TRAFFIC_CONTROL',
    'No Traffic Signs': 'TRAFFIC_CONTROL',
    'Road Lighting Fault': 'TRAFFIC_CONTROL',
    'Damaged Road': 'ROAD_ENVIRONMENT',
    'Damaged/Poor Road Surface': 'ROAD_ENVIRONMENT',
    'Slippery Road': 'ROAD_ENVIRONMENT',
    'Slippery Contaminants on Road': 'ROAD_ENVIRONMENT',
    'Narrow Road': 'ROAD_ENVIRONMENT',
    'Road Geometry Factors': 'ROAD_ENVIRONMENT',
    'Insufficient Roadside Clear Zone': 'ROAD_ENVIRONMENT',
    'Vehicle Equipment Defect': 'VEHICLE_OPS_DEFECT',
    'Brake Failure': 'VEHICLE_OPS_DEFECT',
    'Tire Failure': 'VEHICLE_OPS_DEFECT',
    'Steering System Fault': 'VEHICLE_OPS_DEFECT',
    'Vehicle Electrical Fault': 'VEHICLE_OPS_DEFECT',
    'Engine Malfunction': 'VEHICLE_OPS_DEFECT',
    'Overloading': 'VEHICLE_OPS_DEFECT',
    'Slow-Moving Vehicle': 'VEHICLE_OPS_DEFECT',
    'Unfamiliar Route/Inexperienced': 'EXPERIENCE',
    'Inexperienced Driving': 'EXPERIENCE',
    'Unfamiliar with Left-Side Driving': 'EXPERIENCE',
    'Other': 'OTHER_UNKNOWN',
    'No Road-Condition-Related Cause': 'OTHER_UNKNOWN',
}

# If df_all['PresumedCause'] holds your English labels:
df_motorcycle['PresumedCause_L1'] = df_motorcycle['PresumedCause'].map(eng2cat).fillna('OTHER_UNKNOWN')


cause_raw = df_motorcycle['PresumedCause'].astype('string')
cause_L1 = pd.Series('OTHER_UNKNOWN', index=cause_raw.index, dtype='string')
cause_L1[cause_raw.isna()] = pd.NA

# def assign(label, pattern):
#     # only fill rows not yet classified
#     mask_free = cause_L1.isna() | (cause_L1 == 'OTHER_UNKNOWN')
#     hits = cause_raw.str.contains(pattern, case=False, na=False, regex=True)
#     cause_L1.loc[mask_free & hits] = label

# # ---- high-specificity buckets ----
# assign('IMPAIRMENT', r'เมาสุรา|ดื่มแล้วขับ|สารออกฤทธิ์')
# assign('SPEEDING', r'ขับรถเร็วเกิน')
# assign('DISTRACTION', r'โทรศัพท์|สิ่งรบกวน(ภายใน|ภายนอก)รถ')
# assign('DRIVER_CONDITION', r'หลับใน|เมื่อยล้า|อาการป่วย|ไร้ความสามารถ|โรคประจำตัว|ปัญหาทางสายตา')

# # Manoeuvres: overtake/lane-change/cut-in/sudden brake/tailgating
# assign('MANOEUVRE', r'แซง|เปลี่ยนช่องจราจร|ปาดหน้า|หยุดรถกะทันหัน|ตามกระชั้นชิด|ตัดหน้า')

# # Priority: legal right-of-way & signalling
# assign('YIELD_SIGNAL', r'ไม่ยอมให้รถที่มีสิทธิ์ไปก่อน|ไม่ให้สิทธิ|ฝ่าฝืนสัญญาณไฟ|เครื่องหมายจราจร|ป้ายหยุด|ไม่ให้สัญญาณ(ชะลอ|เลี้ยว|เข้าจอด|ออกจากที่จอด)')

# # Lane discipline: wrong-way etc.
# assign('LANE_DISCIPLINE', r'ย้อนศร|ผิดช่องทาง|คร่อมเส้น|ไม่ขับรถในช่องทางเดินรถซ้ายสุด')

# # Visibility & lighting
# assign('VISIBILITY_LIGHTING', r'แสงสว่างไม่เพียงพอ|ทัศนวิสัย|ระยะ(การ)?มองเห็น(ไม่เพียงพอ|จำกัด)|มีสิ่งบดบัง|ข้ามถนนโดยมีรถจอดหรือวัตถุข้างทางบดบัง')

# # Roadworks / obstruction
# assign('ROADWORKS_OBSTRUCTION', r'กองวัสดุ|สิ่งกีดขวาง|การซ่อม/สร้าง|รถเสียไม่แสดง(เครื่องหมาย|สัญญาณ)')

# # Traffic control infra problems (signals/signs/markings/lighting assets)
# assign('TRAFFIC_CONTROL', r'ระบบสัญญาณไฟจราจรขัดข้อง|ไม่มีระบบสัญญาณไฟจราจร|ป้ายจราจร(ชำรุด|ถูกบดบัง)|เส้นแบ่งทิศทางจราจร(ชำรุด)?|ไม่มี(ป้ายจราจร|เส้นแบ่ง)|ไฟ(ส่องสว่าง|ฟ้าแสงสว่าง)ชำรุด')

# # Vehicle defects & operations (incl. overloading, slow-moving)
# assign('VEHICLE_OPS_DEFECT', r'อุปกรณ์ยานพาหนะบกพร่อง|เบร(ค|ก)ชำรุด|ห้ามล้อ|ยาง(เสื่อมสภาพ|แตก|รถยนต์ชำรุด)|ระบบไฟฟ้า|เครื่องยนต์ขัดข้อง|ระบบบังคับเลี้ยว|บรรทุกเกิน|ยานพาหนะเคลื่อนที่ช้า')

# # Road environment (surface/geometry/clear zone)
# assign('ROAD_ENVIRONMENT', r'ถนน(ลื่น|แคบ|ชำรุด)|ผิวถนน(สภาพแย่|ชำรุด)|คราบสะสม|ทางโค้งอันตราย|ลักษณะของถนน|ระยะปลอดภัยข้างทางไม่เพียงพอ')

# # Experience / unfamiliarity
# assign('EXPERIENCE', r'ไม่คุ้นเคยเส้นทาง|ไม่ชำนาญ|ไม่เป็น|ไม่คุ้นเคยกับการขับขี่ด้านซ้าย')

# # Unknown/other & “no road-condition cause”
# assign('OTHER_UNKNOWN', r'อื่นๆ|ไม่มีมูลเหตุสันนิษฐานที่เกี่ยวข้องด้านสภาพสายทาง')

# df_all['PresumedCause_L1'] = cause_L1

# # (optional) quick check
# # print(df_all['PresumedCause_L1'].value_counts(dropna=False).head(20))


### AccidentType

In [ ]:
import re
import numpy as np
import pandas as pd

# Start with a clean string column for matching
acc_raw = df_motorcycle['AccidentType'].astype('string')

# Initialize collapsed class with default 'Other'
acc_L1 = pd.Series('Other', index=acc_raw.index, dtype='string')
acc_L1[acc_raw.isna()] = pd.NA

# Convenience matcher
def has(pattern):
    return acc_raw.str.contains(pattern, case=False, na=False, regex=True)

# ---- High-specificity classes first ----

# Pedestrian
acc_L1[has(r'คนเดินเท้า|คนข้าม')] = 'Pedestrian'

# Animal
acc_L1[has(r'สัตว์|สุนัข|กระบือ')] = 'Animal'

# Head-on (opposite direction / cross lane / head-on)
acc_L1[has(r'ประสานงา|ทิศทางตรงกันข้าม|สวนเลน|ข้ามเลน')] = 'HeadOn'

# Rear-end
acc_L1[has(r'ชนท้าย')] = 'RearEnd'

# Sideswipe (parallel lanes)
acc_L1[has(r'เฉี่ยวชนรถข้างเคียง|คู่ขนาน')] = 'Sideswipe'

# Side/Angle impacts (intersection angle, side hits)
acc_L1[has(r'ชนเป็นมุม|ชนด้านข้าง|ชนจากฝั่ง(ซ้าย|ขวา)')] = 'SideImpact'

# Turning / U-turn (left/right turns and U-turns)
acc_L1[has(r'เลี้ยว|กลับรถ')] = 'Turning/UTurn'

# Overtake / Lane change / Cut-in
acc_L1[has(r'แซง|เปลี่ยนช่องจราจร|ปาดหน้า')] = 'Overtake/LaneChange'

# Hit object / obstacle / parked vehicle / roadside fixtures / roadworks
acc_L1[has(r'สิ่งกีดขวาง|สิ่งของ|ป้าย|วัสดุ|กองอยู่|ตกหล่น|ยื่นออก|รถที่จอด|'
           r'เสาไฟ|เสาหลัก|การ์ดเรล|คอสะพาน|ฟุตบาท|กำแพง|เกาะกลาง|งานทาง|ทางเชื่อม')] = 'HitObject'

# Loss of control / rollover / fell / ran off road / skid / ditch
acc_L1[has(r'เสียหลัก|เสียการควบคุม|พลิกคว่ำ|ล้มเอง|ตกถนน|หลุดโค้ง|ตกคู|รื่(น|่)ไถล|ลงข้างทาง')] = 'LossOfControl'

# No details
acc_L1[has(r'ไม่มีรายละเอียด')] = 'NoDetails/Other'

# Assign back
df_motorcycle['AccidentType_L1'] = acc_L1

# --- Sanity checks ---
print("Unique (raw):", df_motorcycle['AccidentType'].nunique(dropna=True))
print("Unique (L1):", df_motorcycle['AccidentType_L1'].nunique(dropna=True))

# Quick frequency table
freq = (df_motorcycle['AccidentType_L1']
        .value_counts(dropna=False)
        .rename_axis('AccidentType_L1')
        .reset_index(name='count'))
freq['pct'] = (freq['count'] / freq['count'].sum() * 100).round(1)
print(freq.head(20))


# Plot the important features

### First Vehicle

In [ ]:
counts = df_motorcycle['FirstVehicle'].value_counts().reset_index()
counts.columns = ['FirstVehicle', 'Count']
# Print the results using tab-delimited format for easy Excel copy-pasting
print(counts.to_csv(index=False, sep='\t'))

In [ ]:
# Count the occurrences of each value in the "FirstVehicle" column
first_vehicle_counts = df_motorcycle['FirstVehicle'].value_counts().sort_index()

plt.figure(figsize=(10, 6))
first_vehicle_counts.plot(kind='bar', color='steelblue')
# plt.title('Distribution of First Vehicle Involved')
plt.xlabel('First Vehicle Type', fontsize=14)
plt.ylabel('Count', fontsize=14)
plt.xticks(fontsize=18)
plt.yticks(fontsize=12)
plt.tight_layout()

# Save the plot to the OUTPUT_DIR
plt.savefig(os.path.join("../data/fig", "first_vehicle_counts.png"), dpi=300)
plt.show()

### AccidentLocation 

In [ ]:
# Count the occurrences of each value in the "AccidentLocation" column
first_vehicle_counts = df_motorcycle['AccidentLocation_tags'].value_counts().sort_index()

plt.figure(figsize=(14, 6))
first_vehicle_counts.plot(kind='bar', color='steelblue')

# plt.title('AccidentLocation')
# plt.xlabel('AccidentLocation')
# plt.ylabel('Count')
# plt.tight_layout()

plt.xlabel('Accident Location', fontsize=16)
plt.ylabel('Count', fontsize=14)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.tight_layout()

# Save the plot to the OUTPUT_DIR
# plt.savefig(os.path.join(OUTPUT_DIR, "first_vehicle_distribution.png"), dpi=300)
plt.show()

In [ ]:
# One-hot from the comma-separated tag string
ml = df_motorcycle['AccidentLocation_tags'].fillna('').str.get_dummies(sep=',')

# (Optional) drop OTHER if you don’t want it in plots
if 'OTHER' in ml.columns:
    ml = ml.drop(columns=['OTHER'])

tag_counts = ml.sum().sort_values(ascending=True)  # asc for nicer horiz bars

plt.figure(figsize=(10, max(5, 0.3*len(tag_counts))))
tag_counts.plot(kind='barh')
# plt.title('Accident Location - Tag Prevalence')
plt.xlabel('Count', fontsize=12)
plt.ylabel('Tag', fontsize=12)
plt.xticks(fontsize=12)
plt.yticks(fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join("../data/fig", "accident_location_tags.png"), dpi=300)
plt.show()


In [ ]:
# Choose top N tags to keep the chart readable
top_n = 12
top_tags = tag_counts.sort_values(ascending=False).head(top_n).index.tolist()

sev_cols = ['Fatalities', 'SeriousInjuries', 'MinorInjuries']
cb_colors = ["#E69F00", "#56B4E9", "#009E73"]  # Fatalities, Serious, Minor

# Aggregate severities per tag (sum across rows where the tag = 1)
severity_by_tag = []
for tag in top_tags:
    mask = ml[tag] == 1
    severity_by_tag.append(df_motorcycle.loc[mask, sev_cols].sum())
severity_by_tag = pd.DataFrame(severity_by_tag, index=top_tags)

# Plot stacked counts
ax = severity_by_tag.plot(kind='bar', stacked=True, figsize=(12, 6), color=cb_colors)
ax.set_title('Severity by Accident Location Tag (Top tags)')
ax.set_xlabel('Tag')
ax.set_ylabel('Count')
ax.legend(title='Severity')

# Add % labels inside each stacked bar
row_pct = severity_by_tag.div(severity_by_tag.sum(axis=1), axis=0) * 100
for i, container in enumerate(ax.containers):  # 0=Fatalities,1=Serious,2=Minor
    for j, bar in enumerate(container):
        h = bar.get_height()
        if h > 0:
            pct = row_pct.iloc[j, i]
            ax.text(
                bar.get_x() + bar.get_width()/2,
                bar.get_y() + h/2,
                f"{pct:.0f}%",
                ha="center", va="center", fontsize=8, color="white"
            )

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np

# Limit to the same top tags (or pick another set)
M = ml[top_tags].astype(int)

# Pairwise co-occurrence counts
co = M.T @ M  # square matrix

# Jaccard similarity = co / (sum_i + sum_j - co)
s = M.sum(axis=0).values
den = (s[:, None] + s[None, :] - co.values).astype(float)
jac = np.divide(co.values, den, out=np.zeros_like(co.values, dtype=float), where=den>0)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(jac, aspect='auto')
ax.set_xticks(range(len(top_tags))); ax.set_xticklabels(top_tags, rotation=45, ha='right')
ax.set_yticks(range(len(top_tags))); ax.set_yticklabels(top_tags)
ax.set_title('Accident Location Tag Co-occurrence (Jaccard)')
fig.colorbar(im, ax=ax, label='Jaccard similarity')
plt.tight_layout()
plt.show()


### PresumedCause

In [ ]:
counts = df_motorcycle['PresumedCause_L1'].value_counts().reset_index()
counts.columns = ['PresumedCause_L1', 'Count']
# Print the results using tab-delimited format for easy Excel copy-pasting
print(counts.to_csv(index=False, sep='\t'))

In [ ]:
# Count the occurrences of each value in the "PresumedCause_L1" column
first_vehicle_counts = df_motorcycle['PresumedCause_L1'].value_counts().sort_index()

plt.figure(figsize=(10, 6))
first_vehicle_counts.plot(kind='bar', color='steelblue')
# plt.title('PresumedCause')
# plt.xlabel('Presumed Cause')
# plt.ylabel('Count')
# plt.tight_layout()

plt.xlabel('Presumed Cause', fontsize=14)
plt.ylabel('Count', fontsize=14)
plt.xticks(fontsize=16)
plt.yticks(fontsize=12)
plt.tight_layout()

# Save the plot to the OUTPUT_DIR
# plt.savefig(os.path.join("../data/fig", "PresumedCause_counts.png"), dpi=300)
plt.show()

### AccidentType

In [ ]:
counts = df_motorcycle['AccidentType_L1'].value_counts().reset_index()
counts.columns = ['AccidentType_L1', 'Count']
# Print the results using tab-delimited format for easy Excel copy-pasting
print(counts.to_csv(index=False, sep='\t'))

In [ ]:
# Count the occurrences of each value in the "AccidentType_L1" column
AccidentType_L1_counts = df_motorcycle['AccidentType_L1'].value_counts().sort_index()

plt.figure(figsize=(10, 6))
AccidentType_L1_counts.plot(kind='bar', color='steelblue')
# plt.title('AccidentType')
# plt.xlabel('AccidentType')
# plt.ylabel('Count')
# plt.tight_layout()

plt.xlabel('Accident Type', fontsize=14)
plt.ylabel('Count', fontsize=14)
plt.xticks(fontsize=18)
plt.yticks(fontsize=12)
plt.tight_layout()

# Save the plot to the OUTPUT_DIR
# plt.savefig(os.path.join("../data/fig", "AccidentType_counts.png"), dpi=300)
plt.show()

### Weather

In [ ]:
# print unique values in weather conditions
unique_weather = df_motorcycle['Weather'].unique()
print("Unique Weather Conditions:", unique_weather)

In [ ]:
# translate weather conditions to English
weather_translation = {
    'แจ่มใส': 'Clear',
    'มืดครึ้ม': 'Gloomy',
    'อื่นๆ': 'Other',
    'มีหมอก/ควัน/ฝุ่น': 'Fog/Smoke/Dust',
    'ฝนตก': 'Rain',
    'ภัยธรรมชาติ เช่น พายุ น้ำท่วม': 'Natural Disaster (Storm/Flood)'
}
df_motorcycle['Weather'] = df_motorcycle['Weather'].replace(weather_translation)

In [ ]:
counts = df_motorcycle['Weather'].value_counts().reset_index()
counts.columns = ['Weather', 'Count']
# Print the results using tab-delimited format for easy Excel copy-pasting
print(counts.to_csv(index=False, sep='\t'))

In [ ]:
# Count the occurrences of each value in the "Weather" column
weather_counts = df_motorcycle['Weather'].value_counts().sort_index()

plt.figure(figsize=(10, 6))
weather_counts.plot(kind='bar', color='steelblue')
# plt.title('Weather')
# plt.xlabel('Weather')
# plt.ylabel('Count')
# plt.tight_layout()

plt.xlabel('Weather', fontsize=14)
plt.ylabel('Count', fontsize=14)
plt.xticks(fontsize=16)
plt.yticks(fontsize=12)
plt.tight_layout()

# Save the plot to the OUTPUT_DIR
plt.savefig(os.path.join("../data/fig", "Weather_counts.png"), dpi=300)
plt.show()

### Table to summarise 
% FirstVehicle % AccidentLocation % PresumedCause % AccidentType % Weather

In [ ]:
# print unique values in FirstVehicle
print("Unique values in 'FirstVehicle':", df_motorcycle['FirstVehicle'].unique())
print("Unique values in 'AccidentLocation_tags':", df_motorcycle['AccidentLocation_tags'].unique())
print("Unique values in 'PresumedCause':", df_motorcycle['PresumedCause'].unique())
print("Unique values in 'PresumedCause_L1':", df_motorcycle['PresumedCause_L1'].unique())
print("Unique values in 'AccidentType_L1':", df_motorcycle['AccidentType_L1'].unique())

In [ ]:
df_motorcycle
# save df_motorcycle as csv
df_motorcycle.to_csv(os.path.join("../data/", "TRAMS_motorcycle_2019-2024.csv"), index=False)

## Single-motorcycle crashes 

In [ ]:
# Filter for rows where only motorcycle is involved (all other vehicle columns = 0)
# and total vehicles involved is 1
df_motorcycle_single = df_motorcycle[
    (df_motorcycle['MotorTricycle'] == 0) &
    (df_motorcycle['PrivateCar'] == 0) &
    (df_motorcycle['Van'] == 0) &
    (df_motorcycle['PickupPassenger'] == 0) &
    (df_motorcycle['BusOver4Wheels'] == 0) &
    (df_motorcycle['PickupTruck4Wheels'] == 0) &
    (df_motorcycle['Truck6Wheels'] == 0) &
    (df_motorcycle['TruckUpTo10Wheels'] == 0) &
    (df_motorcycle['TruckOver10Wheels'] == 0) &
    (df_motorcycle['E-TanTruck'] == 0) &
    (df_motorcycle['OtherVehicles'] == 0) &
    (df_motorcycle['Pedestrian'] == 0) &
    # (df_motorcycle['Bicycle'] == 0) &
    (df_motorcycle['VehiclesInvolved'] == 1)
]

# Display info about the single motorcycle accidents
print(f"Number of single motorcycle accidents: {df_motorcycle_single.shape[0]}")
print(f"Percentage of motorcycle accidents: {df_motorcycle_single.shape[0]/df_motorcycle.shape[0]*100:.2f}%")

# Single motorcycle accidents by year
single_by_year = df_motorcycle_single.groupby('Year').size()
total_by_year = df_motorcycle.groupby('Year').size()
percentage_by_year = (single_by_year / total_by_year * 100).round(2)

print("\nSingle motorcycle accidents by year:")
result_df = pd.DataFrame({
    'Single Motorcycle Accidents': single_by_year,
    'Total Motorcycle Accidents': total_by_year,
    'Percentage (%)': percentage_by_year
})
print(result_df)

# Multi-vehicle motorcycle crashes
multi_motorcycle = df_motorcycle[df_motorcycle['VehiclesInvolved'] > 1]

# Count occurrences of each vehicle type in multi-vehicle motorcycle crashes
vehicle_columns = ['PrivateCar', 'PickupTruck4Wheels', 'Van', 'PickupPassenger', 
                  'BusOver4Wheels', 'Truck6Wheels', 'TruckUpTo10Wheels', 
                  'TruckOver10Wheels', 'MotorTricycle', 'E-TanTruck', 'OtherVehicles', 'Pedestrian','Motorcycle']

# Calculate how many accidents involve each vehicle type
vehicle_involvement = {}
for col in vehicle_columns:
    vehicle_involvement[col] = multi_motorcycle[multi_motorcycle[col] > 0].shape[0]

# Convert to DataFrame for easier visualization
vehicle_df = pd.DataFrame(list(vehicle_involvement.items()), columns=['VehicleType', 'Count'])
vehicle_df['Percentage'] = vehicle_df['Count'] / multi_motorcycle.shape[0] * 100
vehicle_df = vehicle_df.sort_values('Count', ascending=False)

print("\nOther vehicles involved in multi-vehicle motorcycle crashes:")
print(vehicle_df)

# Visualize the distribution of single vs. multi-vehicle motorcycle accidents by year
plt.figure(figsize=(12, 6))
bar_width = 0.35
x = np.arange(len(single_by_year.index))

plt.bar(x - bar_width/2, single_by_year.values, bar_width, label='Single Motorcycle')
plt.bar(x + bar_width/2, (total_by_year - single_by_year).values, bar_width, label='Multi-Vehicle')

plt.xlabel('Year', fontsize=14)
plt.ylabel('Number of Accidents', fontsize=14)
plt.title('Single vs. Multi-Vehicle Motorcycle Accidents by Year')
plt.xticks(x, single_by_year.index, fontsize=12)
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()

plt.savefig(os.path.join("../data/fig", "single_vs_multi_motorcycle_by_year.png"), dpi=300)
plt.show()

# Visualize the results of vehicle involvement
plt.figure(figsize=(12, 6))
colors = plt.cm.tab20(np.linspace(0, 1, len(vehicle_df)))
vehicle_df.plot(x='VehicleType', y='Count', kind='bar', color=colors)
plt.title('Other Vehicles Involved in Multi-Vehicle Motorcycle Crashes')
plt.xlabel('Vehicle Type', fontsize=14)
plt.ylabel('Number of Crashes', fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=12)
plt.tight_layout()
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.savefig(os.path.join("../data/fig", "vehicles_involved_with_motorcycles.png"), dpi=300)
plt.show()

In [ ]:
df_motorcycle

In [ ]:
# --- Separate categories ---
# Single motorcycle crashes
single_mc = df_motorcycle_single

# Multi-motorcycle only (≥2 motorcycles, no other vehicle types)
multi_mc_only = df_motorcycle[
    (df_motorcycle['VehiclesInvolved'] > 1) &
    (df_motorcycle[[
        'PrivateCar','PickupTruck4Wheels','Van','PickupPassenger','BusOver4Wheels',
        'Truck6Wheels','TruckUpTo10Wheels','TruckOver10Wheels','MotorTricycle',
        'E-TanTruck','OtherVehicles','Pedestrian'
    ]].sum(axis=1) == 0)  # ensure no non-motorcycle vehicles
]

# Motorcycle + other vehicles
multi_mc_other = df_motorcycle[
    (df_motorcycle['VehiclesInvolved'] > 1) &
    (df_motorcycle[[
        'PrivateCar','PickupTruck4Wheels','Van','PickupPassenger','BusOver4Wheels',
        'Truck6Wheels','TruckUpTo10Wheels','TruckOver10Wheels','MotorTricycle',
        'E-TanTruck','OtherVehicles','Pedestrian'
    ]].sum(axis=1) > 0)
]

# --- Count by year ---
single_by_year = single_mc.groupby('Year').size()
multi_mc_only_by_year = multi_mc_only.groupby('Year').size()
multi_mc_other_by_year = multi_mc_other.groupby('Year').size()

# # Ensure alignment of all years
# years = sorted(df_motorcycle['Year'].unique())
# single_by_year = single_by_year.reindex(years, fill_value=0)
# multi_mc_only_by_year = multi_mc_only_by_year.reindex(years, fill_value=0)
# multi_mc_other_by_year = multi_mc_other_by_year.reindex(years, fill_value=0)

# # --- Plot ---
# plt.figure(figsize=(12, 6))
# bar_width = 0.25
# x = np.arange(len(years))

# plt.bar(x - bar_width, single_by_year.values, bar_width, label='Single Motorcycle', color='tab:blue')
# plt.bar(x, multi_mc_only_by_year.values, bar_width, label='Motorcycle-Motorcycle', color='tab:orange')
# plt.bar(x + bar_width, multi_mc_other_by_year.values, bar_width, label='Motorcycle + Other Vehicles', color='tab:red')

# plt.xlabel('Year', fontsize=14)
# plt.ylabel('Number of Accidents', fontsize=14)
# plt.title('Single vs Multi-Motorcycle vs Motorcycle + Other Vehicles Crashes by Year')
# plt.xticks(x, years, fontsize=12)
# plt.legend()
# plt.grid(axis='y', linestyle='--', alpha=0.7)
# plt.tight_layout()

# plt.savefig(os.path.join(
#     "../data/fig",
#     "single_vs_multi_motorcycle_split_by_year.png"
# ), dpi=300)
# plt.show()


# import numpy as np
# import matplotlib.pyplot as plt

# --- prep: make sure you have these from your earlier code ---
# multi_mc_only: df with >=2 motorcycles and no other vehicles
# multi_mc_other: df with motorcycles + at least one non-motorcycle vehicle
# vehicle_df: DataFrame with columns ['VehicleType','Count'] for multi-vehicle crashes
#             and includes a row 'Motorcycle' with the total count of motorcycle-involved multivehicle crashes

# --- prep counts ---
single_mc_count     = df_motorcycle_single.shape[0]    # single motorcycle
multi_mc_only_count = multi_mc_only.shape[0]           # motorcycle-motorcycle
multi_mc_other_count = multi_mc_other.shape[0]         # motorcycle + other

# Build base vehicle counts (multi-vehicle crashes)
vd = vehicle_df.set_index("VehicleType").copy()
x_labels = vd.index.tolist()
x = np.arange(len(x_labels))

# Ensure Motorcycle is included
if "Motorcycle" not in vd.index:
    raise ValueError("'Motorcycle' not found in vehicle_df")

# total motorcycle-involved = sum of 3 components
mc_total = single_mc_count + multi_mc_only_count + multi_mc_other_count

# --- plot ---
fig, ax = plt.subplots(figsize=(9, 6))

# Draw all vehicle bars first (neutral color)
bars = ax.bar(x, vd["Count"].values, color="#9ecae1", edgecolor="black", linewidth=0.6, label="Other Vehicles")

# Index of Motorcycle
i_mc = x_labels.index("Motorcycle")

# Overwrite Motorcycle bar with stacked version
ax.bar(x[i_mc], single_mc_count, color="tab:blue", edgecolor="black", linewidth=0.6,
       label="Single Motorcycle")
ax.bar(x[i_mc], multi_mc_only_count, bottom=single_mc_count,
       color="tab:orange", edgecolor="black", linewidth=0.6,
       label="Motorcycle-Motorcycle")
ax.bar(x[i_mc], multi_mc_other_count, bottom=single_mc_count+multi_mc_only_count,
       color="tab:red", edgecolor="black", linewidth=0.6,
       label="Motorcycle + Other Vehicles")

# Cosmetics
ax.set_title("Vehicles Involved in Motorcycle Crashes (Single & Multi-Vehicle)")
ax.set_xlabel("Vehicle Type")
ax.set_ylabel("Number of Crashes")
ax.set_xticks(x)
ax.set_xticklabels(x_labels, rotation=45, ha="right")
ax.grid(axis="y", linestyle="--", alpha=0.6)

# Legend without duplicates
handles, labels = ax.get_legend_handles_labels()
unique = dict(zip(labels, handles))
ax.legend(unique.values(), unique.keys(), frameon=True)

plt.tight_layout()
plt.savefig("../data/fig/vehicles_involved_with_motorcycles_split3.png",
            dpi=300)
plt.show()

In [ ]:
print(f"Number of rows where Motorcycle > 1: {(df_motorcycle['Motorcycle'] > 1).sum()}")

In [ ]:
# Group multi-vehicle motorcycle crashes by year and vehicle type
multi_vehicle_by_year_type = multi_motorcycle.groupby(['Year', 'FirstVehicle']).size().unstack(fill_value=0)

# Plot the results
plt.figure(figsize=(12, 8))
multi_vehicle_by_year_type.plot(kind='bar', stacked=True, alpha=0.8, colormap='tab20', ax=plt.gca())

plt.title('Multi-Vehicle Motorcycle Crashes by Year and Vehicle Type', fontsize=16)
plt.xlabel('Year', fontsize=14)
plt.ylabel('Number of Crashes', fontsize=14)
plt.xticks(rotation=45, fontsize=12)
plt.yticks(fontsize=12)
plt.legend(title='Vehicle Type', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()

plt.show()


## Severity and vehicle type
(Fatalities, SeriousInjuries,MinorInjuries) (Motorcycle,MotorTricycle,PrivateCar,Van, PickupPassenger,BusOver4Wheels,PickupTruck4Wheels,Truck6Wheels,TruckUpTo10Wheels,TruckOver10Wheels, E-TanTruck,OtherVehicles,Pedestrian )

In [ ]:
yearly_severity = df_motorcycle.groupby('Year')[['Fatalities', 'SeriousInjuries', 'MinorInjuries']].sum()
# Plot the data
plt.figure(figsize=(10, 6))
yearly_severity.plot(kind='bar', stacked=True, alpha=0.8, color=["red", "orange", "blue"], ax=plt.gca())

# Add labels and title
plt.title('Fatalities, Serious Injuries, and Minor Injuries by Year', fontsize=16)
plt.xlabel('Year', fontsize=12)
plt.ylabel('Number of Casualties', fontsize=12)
plt.xticks(rotation=0, fontsize=10)
plt.yticks(fontsize=10)
plt.legend(title="Severity", fontsize=10)

# Display the plot
plt.tight_layout()
plt.savefig(os.path.join("../data/fig", "yearly_severity_motorcycle.png"), dpi=300)
plt.show()


In [ ]:
# List of vehicle columns to plot
vehicle_columns = [
    "Motorcycle", "MotorTricycle", "PrivateCar", "Van", "PickupPassenger",
    "BusOver4Wheels", "PickupTruck4Wheels", "Truck6Wheels",
    "TruckUpTo10Wheels", "TruckOver10Wheels", "E-TanTruck", "OtherVehicles", "Pedestrian"
]

# Group by Year and sum the counts for each vehicle type
vehicle_by_year = df_motorcycle.groupby("Year")[vehicle_columns].sum()

# Calculate percentages for each vehicle type per year
vehicle_by_year_pct = vehicle_by_year.div(vehicle_by_year.sum(axis=1), axis=0) * 100

# Plot the data
plt.figure(figsize=(12, 8))
ax = vehicle_by_year.plot(kind="bar", stacked=True, colormap="tab20c", ax=plt.gca())

# Add labels and title
plt.title("Vehicle Types Involved in Motorcycle Accidents by Year", fontsize=16)
plt.xlabel("Year", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.xticks(rotation=0, fontsize=10)
plt.yticks(fontsize=10)
plt.legend(title="Vehicle Type", bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=10)

# Add percentage labels to each segment
for i, container in enumerate(ax.containers):
    for bar in container:
        height = bar.get_height()
        if height > 0:
            year_idx = int(bar.get_x() + bar.get_width() / 2)
            vehicle_type = vehicle_by_year.columns[i]
            pct = vehicle_by_year_pct.iloc[year_idx, i]
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_y() + height / 2,
                f"{pct:.1f}%",
                ha="center", va="center", fontsize=10, color="black"
            )

# Display the plot
plt.tight_layout()
plt.savefig(os.path.join("../data/fig", "vehicle_types_involved_with_motorcycles_by_year.png"), dpi=300)
plt.show()


# Spatial

In [ ]:
import geopandas as gpd
import contextily as ctx
import matplotlib.pyplot as plt
import osmnx as ox
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
from matplotlib.font_manager import FontProperties
from matplotlib.patches import Rectangle

# ---------- helpers ----------
def _nice_length_m(data_width_m):
    """Pick a nice scalebar length (1/2/5 × 10^n) ~ 1/5 of axis width."""
    target = data_width_m / 5.0
    import math
    exp = int(math.floor(math.log10(target))) if target > 0 else 0
    base = target / (10 ** exp)
    for b in (1, 2, 5, 10):
        if base <= b:
            return b * (10 ** exp)
    return 10 * (10 ** exp)

def add_scalebar(ax, loc="lower left", font_size=9, pad=0.02):
    """Add a scalebar (meters) to an EPSG:3857 axis."""
    xmin, xmax = ax.get_xlim()
    length_m = _nice_length_m(max(xmax - xmin, 1))  # avoid zero width
    label = f"{int(length_m/1000)} km" if length_m >= 1000 else f"{int(length_m)} m"
    fp = FontProperties(size=font_size)
    sb = AnchoredSizeBar(
        ax.transData, length_m, label, loc,
        pad=pad, color='black', frameon=False,
        size_vertical=max((xmax - xmin), 1) * 0.003,
        fontproperties=fp
    )
    ax.add_artist(sb)

def add_north_arrow(ax, xy=(0.08, 0.82), size=0.10, text="N", text_size=10):
    """Add a simple north arrow in axes-fraction coords (0..1)."""
    ax.annotate(
        "", xy=(xy[0], xy[1] + size), xytext=xy,
        xycoords="axes fraction", textcoords="axes fraction",
        arrowprops=dict(arrowstyle="-|>", linewidth=1.5, color="black")
    )
    ax.text(
        xy[0], xy[1] + size + 0.015, text,
        transform=ax.transAxes, ha="center", va="bottom",
        fontsize=text_size, fontweight="bold"
    )

# ---------- boundaries ----------
th_boundary = ox.geocode_to_gdf("Thailand").to_crs(4326)
bkk_boundary = ox.geocode_to_gdf("Bangkok, Thailand").to_crs(4326)

# ---------- motorcycle points to GeoDataFrame ----------
gdf = gpd.GeoDataFrame(
    df_motorcycle,
    geometry=gpd.points_from_xy(df_motorcycle["Longitude"], df_motorcycle["Latitude"]),
    crs="EPSG:4326",
)

# ---------- clip ----------
gdf_th = gpd.clip(gdf, th_boundary)
gdf_bkk = gpd.clip(gdf, bkk_boundary)

# ---------- reproject to Web Mercator ----------
gdf_th_3857       = gdf_th.to_crs(3857)
gdf_bkk_3857      = gdf_bkk.to_crs(3857)
th_boundary_3857  = th_boundary.to_crs(3857)
bkk_boundary_3857 = bkk_boundary.to_crs(3857)

# ---------- main figure (Thailand) ----------
fig, ax = plt.subplots(figsize=(12, 8))
th_boundary_3857.boundary.plot(ax=ax, linewidth=0.8, alpha=0.6)
gdf_th_3857.plot(ax=ax, markersize=.2, color="tab:blue", alpha=0.2, label="Thailand")
gdf_bkk_3857.plot(ax=ax, markersize=.2, color="tab:red",  alpha=0.2, label="Bangkok")
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, attribution_size=5, zorder=0)
ax.legend(loc="upper right")
ax.set_axis_off()

# add north arrow + scalebar to the main map
add_north_arrow(ax, xy=(0.88, 0.75), size=0.10)
add_scalebar(ax, loc="lower right")

# ---------- inset map (Bangkok zoom-in, cropped) ----------
axins = inset_axes(ax, width="55%", height="55%", loc="lower right", borderpad=1)

# crop: remove bottom 40% of Bangkok extent
minx, miny, maxx, maxy = bkk_boundary_3857.total_bounds
crop_ymin = miny + 0.4 * (maxy - miny)

# inset layers
bkk_boundary_3857.boundary.plot(ax=axins, linewidth=1.0, color="black", zorder=2)
gdf_bkk_3857.plot(ax=axins, markersize=2, color="tab:red", alpha=0.7, zorder=3)
ctx.add_basemap(axins, source=ctx.providers.CartoDB.Positron, attribution_size=5, zorder=0)

# apply cropped view
axins.set_xlim(minx, maxx)
axins.set_ylim(crop_ymin, maxy)

axins.set_xticks([])
axins.set_yticks([])
axins.set_xticklabels([])
axins.set_yticklabels([])

# add visible border to inset itself
for spine in axins.spines.values():
    spine.set_visible(True)
    spine.set_edgecolor("black")
    spine.set_linewidth(1.5)
    spine.set_zorder(10)

# scalebar on inset (north arrow only on main map)
add_scalebar(axins, loc="lower right")

# ---------- rectangle on main map showing inset extent ----------
rect = Rectangle(
    (minx, crop_ymin),  # lower-left corner
    (maxx - minx),      # width
    (maxy - crop_ymin), # height
    fill=False, edgecolor="black", linewidth=1, linestyle="-", zorder=20
)
ax.add_patch(rect)

plt.tight_layout()
# tight save with no boarder
plt.savefig(os.path.join("../data/fig", "motorcycle_accidents_map.pdf"), dpi=300, bbox_inches='tight', pad_inches=0)
plt.show()


In [ ]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import shapely
# --- paths (adjust if needed) ---
base = Path("../data/boundary")
p_prov   = base / "TH_Province.shp"      # provinces (Changwat)
p_amphoe = base / "Amphoe_PROV.shp"      # districts (Amphoe / Khet)
p_tambon = base / "TH_Tambon.shp"        # subdistricts (Tambon / Khwaeng)

OUT_GPKG = base / "th_admin.gpkg"

def read_fix(path):
    # try utf-8, fall back to tis-620 if Thai labels break
    try:
        gdf = gpd.read_file(path)
    except UnicodeDecodeError:
        gdf = gpd.read_file(path, encoding="tis-620")
    # fix invalids
    gdf = gdf[gdf.geometry.notnull()].copy()
    gdf["geometry"] = gdf.geometry.apply(lambda g: shapely.make_valid(g) if g is not None and not g.is_valid else g)
    gdf = gdf[~gdf.is_empty]
    # enforce WGS84
    if gdf.crs is None:
        # many Thai admin layers come in EPSG:4326; set if missing
        gdf.set_crs(4326, inplace=True)
    else:
        gdf = gdf.to_crs(4326)
    return gdf

# --- read layers ---
provinces = read_fix(p_prov)
amphoe    = read_fix(p_amphoe)
tambon    = read_fix(p_tambon)

print(f"provinces: {len(provinces)}  crs={provinces.crs}")
print(f"amphoe:    {len(amphoe)}      crs={amphoe.crs}")
print(f"tambon:    {len(tambon)}      crs={tambon.crs}")

# --- harmonise columns (rename to consistent English keys if present) ---
# Adjust these mappings to your actual field names (use provinces.columns to inspect)
def rename_cols(gdf, mapping_options):
    cols = gdf.columns
    mapping = {}
    for std_key, candidates in mapping_options.items():
        for c in candidates:
            if c in cols:
                mapping[c] = std_key
                break
    return gdf.rename(columns=mapping)

prov_map = {
    "prov_th": ["PROV_NAMT","PROV_TH","prov_name","Province_T","PROV_NAME_T"],
    "prov_en": ["PROV_NAME","PROV_ENG","Province_E","PROV_NAME_E","NAME_ENG"],
    "prov_id": ["PROV_CODE","PROV_ID","PROV_IDN","CC_1","ADM1_PCODE","ADM1_EN"]
}
amp_map = {
    "prov_id": ["PROV_CODE","PROV_ID","CC_1","ADM1_PCODE"],
    "prov_en": ["TH_Provi_1","PROV_ENG","Province_E","PROV_NAME_E","NAME_ENG"],
    "amp_th":  ["AMP_NAMT","AMPH_NAMT","AMPHOE_T","AMPHUR_T","AMP_NAM_T","ADM2_TH"],
    "amp_en":  ["AMP_NAMT_E","AMP_NAME_E","AMPHOE_E","AMPHUR_E","AMP_NAM_E","ADM2_EN"],
    "amp_id":  ["AMP_CODE","AMP_ID","ADM2_PCODE","DIST_CODE"]
}
tam_map = {
    "prov_id": ["PROV_CODE","CC_1","ADM1_PCODE"],
    "amp_id":  ["AMP_CODE","ADM2_PCODE","DIST_CODE"],
    "tam_th":  ["TAM_NAMT","TAMBON_T","TAM_NAM_T","ADM3_TH"],
    "tam_en":  ["TAM_NAM_E","TAMBON_E","TAM_NAM_E","ADM3_EN"],
    "tam_id":  ["TAM_CODE","ADM3_PCODE","SUBDIST_CODE"]
}

provinces = rename_cols(provinces, prov_map)
amphoe    = rename_cols(amphoe, amp_map)
tambon    = rename_cols(tambon, tam_map)

# --- optional: keep only relevant columns + geometry
provinces = provinces[sorted([c for c in provinces.columns if c in {"prov_th","prov_en","prov_id"}] + ["geometry"])]
amphoe    = amphoe[sorted([c for c in amphoe.columns if c in {"prov_id","prov_en","amp_th","amp_en","amp_id"}] + ["geometry"])]
tambon    = tambon[sorted([c for c in tambon.columns if c in {"prov_id","amp_id","tam_th","tam_en","tam_id"}] + ["geometry"])]

# --- save to GeoPackage and GeoJSONs ---
if OUT_GPKG.exists():
    OUT_GPKG.unlink()
provinces.to_file(OUT_GPKG, layer="provinces", driver="GPKG")
amphoe.to_file(OUT_GPKG, layer="districts", driver="GPKG")
tambon.to_file(OUT_GPKG, layer="subdistricts", driver="GPKG")

provinces.to_file(base / "provinces.geojson", driver="GeoJSON")
amphoe.to_file(base / "districts.geojson", driver="GeoJSON")
tambon.to_file(base / "subdistricts.geojson", driver="GeoJSON")

print("Saved:", OUT_GPKG)


In [ ]:
import geopandas as gpd
import contextily as ctx
import matplotlib.pyplot as plt

# --- CRS alignment ---
provinces = provinces.to_crs(4326)
amphoe    = amphoe.to_crs(4326)
gdf_th    = gdf_th.to_crs(4326)   # motorcycle crash points

# --- IDs as strings ---
provinces["prov_id"] = provinces["prov_id"].astype(str)
amphoe["amp_id"]     = amphoe["amp_id"].astype(str)

# --- PROVINCES: compute counts ---
pts_in_prov = gpd.sjoin(
    gdf_th[["geometry"]],
    provinces[["prov_id", "geometry"]],
    how="left",
    predicate="within",
)

prov_counts = (
    pts_in_prov.groupby("prov_id")
    .size()
    .rename("AccidentCount")
    .reset_index()
)

# drop any existing AccidentCount to avoid suffix clash
provinces = provinces.drop(columns=["AccidentCount"], errors="ignore")
provinces = provinces.merge(prov_counts, on="prov_id", how="left")
provinces["AccidentCount"] = provinces["AccidentCount"].fillna(0).astype(int)

# --- AMPHOE: compute counts ---
pts_in_amp = gpd.sjoin(
    gdf_th[["geometry"]],
    amphoe[["amp_id", "geometry"]],
    how="left",
    predicate="within",
)

amp_counts = (
    pts_in_amp.groupby("amp_id")
    .size()
    .rename("AccidentCount")
    .reset_index()
)

amphoe = amphoe.drop(columns=["AccidentCount"], errors="ignore")
amphoe = amphoe.merge(amp_counts, on="amp_id", how="left")
amphoe["AccidentCount"] = amphoe["AccidentCount"].fillna(0).astype(int)

# --- Reproject for basemap ---
provinces_3857 = provinces.to_crs(3857)
amphoe_3857    = amphoe.to_crs(3857)

# --- Plot with separate colorbars ---
fig, axes = plt.subplots(1, 2, figsize=(14, 7), constrained_layout=True)

# Province heatmap
provinces_3857.plot(
    column="AccidentCount",
    cmap="Reds",
    edgecolor="black",
    linewidth=0.2,
    ax=axes[0],
    legend=True,
    legend_kwds={
        "label": "Crash count (province)",
        "orientation": "vertical",
        "shrink": 0.6,   # shrink colorbar height
        "pad": 0.02      # reduce distance between map and colorbar
    },
)
axes[0].set_title("Motorcycle Crash Count by Province (2019-2024)")
axes[0].axis("off")
ctx.add_basemap(axes[0], source=ctx.providers.CartoDB.Positron)

# Amphoe heatmap
amphoe_3857.plot(
    column="AccidentCount",
    cmap="Reds",
    edgecolor="black",
    linewidth=0.2,
    ax=axes[1],
    legend=True,
    legend_kwds={
        "label": "Crash count (amphoe)",
        "orientation": "vertical",
        "shrink": 0.6,
        "pad": 0.02
    },
)
axes[1].set_title("Motorcycle Crash Count by Amphoe (2019-2024)")
axes[1].axis("off")
ctx.add_basemap(axes[1], source=ctx.providers.CartoDB.Positron)

plt.show()


In [ ]:
import matplotlib as mpl
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# --- Provinces map ---
fig, ax = plt.subplots(figsize=(7, 9))
provinces_3857.plot(
    column="AccidentCount",
    cmap="Reds",
    edgecolor="black",
    linewidth=0.2,
    ax=ax,
    legend=False,   # turn off auto-legend
)
ax.axis("off")
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)
add_north_arrow(ax, xy=(0.88, 0.82), size=0.10)
add_scalebar(ax, loc="lower right")

# custom colorbar inset
sm = mpl.cm.ScalarMappable(
    cmap="Reds",
    norm=mpl.colors.Normalize(
        vmin=provinces_3857["AccidentCount"].min(),
        vmax=provinces_3857["AccidentCount"].max()
    )
)
sm._A = []
cax = inset_axes(ax, width="3%", height="30%", loc="lower right",
                 bbox_to_anchor=(-0.2, 0.05, 1, 1),  # position inside map
                 bbox_transform=ax.transAxes, borderpad=0)
cbar = fig.colorbar(sm, cax=cax)
cbar.set_label("Crash count (province)")

plt.savefig(
    os.path.join("../data/fig",
                 "motorcycle_accidents_Province.png"),
    dpi=300, bbox_inches="tight", pad_inches=0
)
plt.show()


# --- Amphoe map ---
fig, ax = plt.subplots(figsize=(7, 9))
amphoe_3857.plot(
    column="AccidentCount",
    cmap="Reds",
    edgecolor="black",
    linewidth=0.2,
    ax=ax,
    legend=False,   # turn off auto-legend
)
ax.axis("off")
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)
add_north_arrow(ax, xy=(0.88, 0.82), size=0.10)
add_scalebar(ax, loc="lower right")

# custom colorbar inset
sm = mpl.cm.ScalarMappable(
    cmap="Reds",
    norm=mpl.colors.Normalize(
        vmin=amphoe_3857["AccidentCount"].min(),
        vmax=amphoe_3857["AccidentCount"].max()
    )
)
sm._A = []
cax = inset_axes(ax, width="3%", height="30%", loc="lower right",
                 bbox_to_anchor=(-0.15, 0.05, 1, 1),
                 bbox_transform=ax.transAxes, borderpad=0)
cbar = fig.colorbar(sm, cax=cax)
cbar.set_label("Crash count (amphoe)")

plt.savefig(
    os.path.join("../data/fig",
                 "motorcycle_accidents_Amphoe.png"),
    dpi=300, bbox_inches="tight", pad_inches=0
)
plt.show()


In [ ]:
# Top 5 highest AccidentCount in provinces_3857
print("Top 5 provinces with highest AccidentCount:")
print(provinces_3857.nlargest(5, 'AccidentCount'))

# Top 5 lowest AccidentCount in provinces_3857
print("\nTop 5 provinces with lowest AccidentCount:")
print(provinces_3857.nsmallest(5, 'AccidentCount'))

# Top 5 highest AccidentCount in amphoe_3857
print("\nTop 5 amphoe with highest AccidentCount:")
print(amphoe_3857.nlargest(5, 'AccidentCount'))

# Top 5 lowest AccidentCount in amphoe_3857
print("\nTop 5 amphoe with lowest AccidentCount:")
print(amphoe_3857.nsmallest(5, 'AccidentCount'))

# All provinces with 0 AccidentCount
print("\nProvinces with 0 AccidentCount:")
print(provinces_3857[provinces_3857['AccidentCount'] == 0])

# All amphoe with 0 AccidentCount
print("\nAmphoe with 0 AccidentCount:")
print(amphoe_3857[amphoe_3857['AccidentCount'] == 0])


In [ ]:
fig, ax = plt.subplots(figsize=(7, 9))
amphoe_3857[amphoe_3857['AccidentCount'] == 0].plot(
    column="AccidentCount",
    # cmap="Reds",
    edgecolor="black",
    color='lightgrey',
    linewidth=1,
    ax=ax,
    legend=False,   # turn off auto-legend
)
ax.axis("off")
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)
add_north_arrow(ax, xy=(0.88, 0.82), size=0.10)
add_scalebar(ax, loc="lower right")

# custom colorbar inset
# sm = mpl.cm.ScalarMappable(
#     cmap="Reds",
#     norm=mpl.colors.Normalize(
#         vmin=amphoe_3857["AccidentCount"].min(),
#         vmax=amphoe_3857["AccidentCount"].max()
#     )
# )
# sm._A = []
# cax = inset_axes(ax, width="3%", height="30%", loc="lower right",
#                  bbox_to_anchor=(-0.15, 0.05, 1, 1),
#                  bbox_transform=ax.transAxes, borderpad=0)
# cbar = fig.colorbar(sm, cax=cax)
# cbar.set_label("Crash count (amphoe)")

# # plt.savefig(
#     os.path.join("../data/fig",
#                  "motorcycle_accidents_Amphoe.png"),
#     dpi=300, bbox_inches="tight", pad_inches=0
# )
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import contextily as ctx

# Thailand bounds in Web Mercator
th_3857 = th_boundary.to_crs(3857)
minx, miny, maxx, maxy = th_3857.total_bounds

# Point coords (already EPSG:3857)
x = gdf_th_3857.geometry.x
y = gdf_th_3857.geometry.y

fig, ax = plt.subplots(figsize=(10, 10))

# KDE - clip to Thailand and tune bandwidth
sns.kdeplot(
    x=x, y=y,
    fill=True,
    cmap="YlOrRd",
    bw_adjust=0.15,           # ↓ less smoothing (adjust 0.15-0.5)
    thresh=0.01,              # trim very low densities
    levels=40,
    clip=((minx, maxx), (miny, maxy)),   # keep inside Thailand
    ax=ax,
    alpha=0.5
)

# Set correct map extent & aspect
ax.set_xlim(minx, maxx)
ax.set_ylim(miny, maxy)
ax.set_aspect("equal", adjustable="box")  # prevent stretching

# Basemap in Web Mercator
ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)

# ax.set_title("Motorcycle Crash Density Heatmap (Seaborn KDE)")
ax.axis("off")
add_north_arrow(ax, xy=(0.88, 0.82), size=0.10)
add_scalebar(ax, loc="lower right")
plt.tight_layout()
plt.savefig(os.path.join("../data/fig", "motorcycle_accidents_kde.pdf"), dpi=300, bbox_inches='tight', pad_inches=0)
plt.show()


# Temporal Characteristics

In [ ]:
# === Setup ===
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ---------- Config ----------
OUTDIR = "fig/"
os.makedirs(OUTDIR, exist_ok=True)

# ---------- 1) Parse AccidentDate and AccidentTime ----------
def _parse_date(s):
    # common patterns: 'MM/DD/YYYY', 'YYYY-MM-DD', etc.
    return pd.to_datetime(s, errors="coerce", infer_datetime_format=True, dayfirst=False)

def _parse_time(s):
    # handles 'HH:MM', 'H:MM', 'HH:MM:SS'; fallback to NaT
    try:
        return pd.to_datetime(s, errors="coerce").dt.time
    except Exception:
        return pd.NaT

df = gdf_th.copy()

# If date is already datetime, keep; else parse
if not np.issubdtype(df["AccidentDate"].dtype, np.datetime64):
    df["AccidentDate_parsed"] = _parse_date(df["AccidentDate"])
else:
    df["AccidentDate_parsed"] = df["AccidentDate"]

# Parse time (string -> datetime.time)
if df["AccidentTime"].dtype == "O":
    # try fast path for HH:MM
    # split and coerce; if it fails, fallback to general parser
    try:
        hhmm = df["AccidentTime"].str.extract(r"^\s*(\d{1,2}):(\d{2})(?::\d{2})?\s*$").astype(float)
        df["AccidentTime_parsed"] = pd.to_timedelta(hhmm[0]*3600 + hhmm[1]*60, unit="s")
    except Exception:
        t_general = pd.to_datetime(df["AccidentTime"], errors="coerce")
        df["AccidentTime_parsed"] = t_general.dt.hour*3600 + t_general.dt.minute*60
        df["AccidentTime_parsed"] = pd.to_timedelta(df["AccidentTime_parsed"], unit="s")
else:
    # already a time-like? coerce to timedelta hours:minutes
    t_general = pd.to_datetime(df["AccidentTime"], errors="coerce")
    df["AccidentTime_parsed"] = t_general.dt.hour*3600 + t_general.dt.minute*60
    df["AccidentTime_parsed"] = pd.to_timedelta(df["AccidentTime_parsed"], unit="s")

# Build full datetime (assume local date with that time)
df["AccidentDateTime"] = df["AccidentDate_parsed"] + df["AccidentTime_parsed"]
df = df.dropna(subset=["AccidentDateTime"]).copy()

# ---------- 2) Derive temporal features ----------
df["year"]   = df["AccidentDateTime"].dt.year
df["month"]  = df["AccidentDateTime"].dt.month
df["ym"]     = df["AccidentDateTime"].dt.to_period("M").dt.to_timestamp()
df["dow"]    = df["AccidentDateTime"].dt.dayofweek  # Monday=0, Sunday=6
df["hour"]   = df["AccidentDateTime"].dt.hour
df["weekend"] = df["dow"].isin([5, 6])  # Sat/Sun

# Optional: filter the analysis window (2019-2024) if not already filtered
df = df[(df["year"] >= 2019) & (df["year"] <= 2024)].copy()

# Nice labels
dow_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
df["dow_label"] = pd.Categorical(df["dow"].map(dict(enumerate(dow_labels))), categories=dow_labels, ordered=True)

# ---------- 3) Monthly trend (with 3-month rolling avg) ----------
monthly = df.groupby("ym").size().rename("count").reset_index()
monthly["roll3"] = monthly["count"].rolling(3, center=True, min_periods=1).mean()

plt.figure(figsize=(10, 4))
plt.plot(monthly["ym"], monthly["count"], linewidth=1.5, label="Monthly count")
plt.plot(monthly["ym"], monthly["roll3"], linewidth=2.0, linestyle="--", label="3-month avg")
plt.title("Motorcycle crashes per month (2019-2024)")
plt.xlabel("Month")
plt.ylabel("Crashes")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "temporal_monthly_trend.png"), dpi=300)
plt.show()

# ---------- 4) Hour-of-day distribution ----------
hourly = df.groupby("hour").size().reindex(range(24), fill_value=0)
plt.figure(figsize=(8, 3))
plt.bar(hourly.index, hourly.values, width=0.9)
plt.xticks(range(0, 24, 2))
plt.title("Crashes by hour of day")
plt.xlabel("Hour")
plt.ylabel("Crashes")
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "temporal_hour_of_day.png"), dpi=300)
plt.show()

# ---------- 5) Day-of-week distribution ----------
dow_counts = df.groupby("dow_label").size().reindex(dow_labels, fill_value=0)
plt.figure(figsize=(8, 3))
plt.bar(dow_counts.index, dow_counts.values)
plt.title("Crashes by day of week")
plt.xlabel("Day of week")
plt.ylabel("Crashes")
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "temporal_day_of_week.png"), dpi=300)
plt.show()

# ---------- 6) Weekday vs weekend hourly profile ----------
prof = (df.groupby(["weekend", "hour"]).size()
        .rename("count")
        .reset_index())
pivot_prof = prof.pivot(index="hour", columns="weekend", values="count").fillna(0)
pivot_prof = pivot_prof.reindex(range(24), fill_value=0)
plt.figure(figsize=(8, 3))
plt.plot(pivot_prof.index, pivot_prof[False], linewidth=2, label="Weekday")
plt.plot(pivot_prof.index, pivot_prof[True],  linewidth=2, linestyle="--", label="Weekend")
plt.xticks(range(0, 24, 2))
plt.title("Hourly profile: weekday vs weekend")
plt.xlabel("Hour")
plt.ylabel("Crashes")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "temporal_hourly_weekday_weekend.png"), dpi=300)
plt.show()

# ---------- 7) Weekday×Hour heatmap ----------
# Build count matrix
wh = df.groupby(["dow_label", "hour"]).size().rename("count").reset_index()
heat = wh.pivot(index="dow_label", columns="hour", values="count").reindex(dow_labels).reindex(range(24), axis=1).fillna(0)

plt.figure(figsize=(10, 3.8))
sns.heatmap(heat, cmap="Reds", linewidths=0.2)
plt.title("Crash density by day-of-week and hour")
plt.xlabel("Hour")
plt.ylabel("Day of week")
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "temporal_heatmap_dow_hour.png"), dpi=300)
plt.show()

# ---------- 8) Seasonal pattern: monthly by year ----------
mby = df.groupby(["year", "month"]).size().rename("count").reset_index()
# pivot: rows = year, cols = month (1..12)
mby_pvt = mby.pivot(index="year", columns="month", values="count").reindex(columns=range(1,13)).fillna(0)

plt.figure(figsize=(10, 3.8))
sns.heatmap(mby_pvt, cmap="Reds", linewidths=0.2, cbar_kws={"label": "Crashes"})
plt.title("Monthly crash counts by year")
plt.xlabel("Month")
plt.ylabel("Year")
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "temporal_heatmap_month_by_year.png"), dpi=300)
plt.show()


In [ ]:
df_motorcycle

In [ ]:
# how many are ทางหลวง and ทางหลวง from AgencyRoute
df_motorcycle['AgencyRoute'].value_counts()

In [ ]:
# save gdf_th_3857
gdf_th_3857.to_file("DOH/motorcycle_accidents_TRAMS.shp")

## Associations among categorical crash attributes


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency

# ---------------- helpers ----------------
def cramers_v_corrected(ct: pd.DataFrame | np.ndarray) -> float:
    """Bias-corrected Cramér's V (Bergsma 2013)."""
    ct = np.asarray(ct, dtype=float)
    n = ct.sum()
    if n == 0:
        return np.nan
    r, k = ct.shape
    if r < 2 or k < 2:
        return np.nan
    chi2, _, _, _ = chi2_contingency(ct, correction=False)
    phi2 = chi2 / n
    phi2corr = max(0, phi2 - (k - 1) * (r - 1) / (n - 1))
    rcorr = r - (r - 1) ** 2 / (n - 1)
    kcorr = k - (k - 1) ** 2 / (n - 1)
    denom = min((kcorr - 1), (rcorr - 1))
    return np.sqrt(phi2corr / denom) if denom > 0 else np.nan

def pair_cramers_v(x: pd.Series, y: pd.Series):
    """Return (V, p-value, contingency table)."""
    ct = pd.crosstab(x, y, dropna=True)
    if ct.empty or ct.values.sum() == 0 or ct.shape[0] < 2 or ct.shape[1] < 2:
        return np.nan, np.nan, ct
    chi2, p, _, _ = chi2_contingency(ct, correction=False)
    v = cramers_v_corrected(ct)
    return v, p, ct

def chisq_top_residuals(x: pd.Series, y: pd.Series, top: int = 12) -> pd.DataFrame:
    """Top absolute standardized residuals for a pair of categoricals."""
    ct = pd.crosstab(x, y, dropna=True)
    if ct.empty or ct.values.sum() == 0 or ct.shape[0] < 2 or ct.shape[1] < 2:
        return pd.DataFrame(columns=['x_cat','y_cat','std_resid','count','abs_resid','chi2','pval','dof','n'])
    chi2, p, dof, expected = chi2_contingency(ct, correction=False)
    resid = (ct - expected) / np.sqrt(expected)
    out = (
        resid.stack().rename('std_resid')
             .rename_axis(index=['x_cat','y_cat']).reset_index()
    )
    out['count'] = ct.stack().values
    out['abs_resid'] = out['std_resid'].abs()
    out['chi2'] = chi2
    out['pval'] = p
    out['dof'] = dof
    out['n'] = int(ct.values.sum())
    out = out.sort_values('abs_resid', ascending=False).head(top)
    return out[['x_cat','y_cat','std_resid','count','abs_resid','chi2','pval','dof','n']]

def fdr_bh(pvals, alpha=0.05):
    """Benjamini-Hochberg FDR; returns (order_idx, passed_mask, critical_threshold)."""
    p = np.asarray(pvals, dtype=float)
    m = len(p)
    if m == 0:
        return np.array([], dtype=int), np.array([], dtype=bool), np.nan
    order = np.argsort(p)
    ranked = np.arange(1, m + 1)
    thresh = ranked * alpha / m
    passed = np.zeros(m, dtype=bool)
    passed[order] = p[order] <= thresh
    kmax = (np.where(passed[order])[0].max() + 1) if passed.any() else 0
    crit = thresh[kmax - 1] if kmax > 0 else 0.0
    return order, passed, crit

# ---------------- inputs ----------------
single_vars = ["FirstVehicle", "Weather", "PresumedCause_L1", "AccidentType_L1"]

# One-hot for multi-label AccidentLocation_tags (comma-separated)
ml = df_all["AccidentLocation_tags"].fillna("").str.get_dummies(sep=",")
if 'OTHER' in ml.columns:
    ml = ml.drop(columns=['OTHER'])  # optional

# ---------------- 1) Cramér’s V among single-label vars ----------------
V = pd.DataFrame(index=single_vars, columns=single_vars, dtype=float)
P = pd.DataFrame(index=single_vars, columns=single_vars, dtype=float)

for a in single_vars:
    for b in single_vars:
        v, p, _ = pair_cramers_v(df_all[a], df_all[b])
        V.loc[a, b] = v
        P.loc[a, b] = p

print("Cramér's V matrix:\n", V, "\n")
print("p-value matrix:\n", P, "\n")

# Heatmap (matplotlib)
plt.figure(figsize=(6, 5))
plt.imshow(V.astype(float), vmin=0, vmax=1)
plt.xticks(range(len(single_vars)), single_vars, rotation=45, ha='right')
plt.yticks(range(len(single_vars)), single_vars)
plt.title("Cramér's V among categorical variables")
plt.colorbar(label="Cramér's V")
plt.tight_layout()
plt.show()

# ---------------- 2) Top standardized residuals per pair ----------------
all_pairs = []
for i in range(len(single_vars)):
    for j in range(i + 1, len(single_vars)):
        a, b = single_vars[i], single_vars[j]
        top = chisq_top_residuals(df_all[a], df_all[b], top=12)
        if not top.empty:
            top.insert(0, 'pair', f'{a} x {b}')
            all_pairs.append(top)
pairs_top = pd.concat(all_pairs, ignore_index=True) if all_pairs else pd.DataFrame(
    columns=['pair','x_cat','y_cat','std_resid','count','abs_resid','chi2','pval','dof','n']
)
print("\nTop standardized residuals per pair (showing first 20 rows):")
print(pairs_top.head(20))

# ---------------- 3) Tag vs single-label variables ----------------
# pick top-N tags to keep things readable
topN = 20
tag_counts = ml.sum().sort_values(ascending=False)
top_tags = tag_counts.head(topN).index.tolist()
ml_top = ml[top_tags]

# 3a) Cramér’s V for (tag present/absent) vs each single var
rows_v = []
for tag in top_tags:
    for var in single_vars:
        v, p, ct = pair_cramers_v(ml_top[tag].map({1: f'{tag}=1', 0: f'{tag}=0'}), df_all[var])
        rows_v.append({"tag": tag, "var": var, "cramers_v": v, "pval": p, "n": int(ct.values.sum() if not ct.empty else 0)})
assoc_tag_v = pd.DataFrame(rows_v).sort_values(["cramers_v","n"], ascending=[False, False])
print("\nTag vs variable - strongest by Cramér’s V (top 20):")
print(assoc_tag_v.head(20))

# 3b) For each (tag,var), show top residual cells (most enriched/depleted categories)
rows_resid = []
for tag in top_tags:
    for var in single_vars:
        t = chisq_top_residuals(ml_top[tag].map({1: f'{tag}=1', 0: f'{tag}=0'}), df_all[var], top=6)
        if t.empty:
            continue
        t.insert(0, 'tag', tag)
        t.insert(1, 'var', var)
        rows_resid.append(t)

assoc_tag_resid = (pd.concat(rows_resid, ignore_index=True)
                   if rows_resid else
                   pd.DataFrame(columns=['tag','var','x_cat','y_cat','std_resid','count','abs_resid','chi2','pval','dof','n']))
assoc_tag_resid = assoc_tag_resid.sort_values(["abs_resid","count"], ascending=[False, False])

print("\nTag vs variable - top standardized residuals (first 20 rows):")
print(assoc_tag_resid.head(20))

# ---------------- 4) Simple visualization: which tags best explain AccidentType_L1 ----------------
focus = "AccidentType_L1"
top_for_focus = (assoc_tag_v[assoc_tag_v["var"] == focus]
                 .dropna(subset=["cramers_v"])
                 .nlargest(10, "cramers_v")
                 .sort_values("cramers_v"))
plt.figure(figsize=(8, 5))
plt.barh(top_for_focus["tag"], top_for_focus["cramers_v"])
plt.xlabel("Cramér's V (effect size)")
plt.ylabel("AccidentLocation tag")
plt.title(f"Top AccidentLocation tags associated with {focus}")
plt.tight_layout()
plt.show()

# ---------------- 5) (optional) FDR across tag-variable tests ----------------
order, passed, crit = fdr_bh(assoc_tag_v["pval"].dropna().values, alpha=0.05)
assoc_tag_v["_fdr_pass"] = False
assoc_tag_v.loc[assoc_tag_v["pval"].notna(), "_fdr_pass"] = passed
print(f"\nFDR (BH) critical threshold: {crit:.4g}")
print("Tag-variable pairs passing FDR (top 20 by V):")
print(assoc_tag_v[assoc_tag_v["_fdr_pass"]].nlargest(20, "cramers_v"))


## Vehicle types

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---- Vehicle columns from your table ----
vehicle_cols = [
    "Motorcycle", "MotorTricycle", "PrivateCar", "Van", "PickupPassenger",
    "BusOver4Wheels", "PickupTruck4Wheels", "Truck6Wheels",
    "TruckUpTo10Wheels", "TruckOver10Wheels", "E-TanTruck", "OtherVehicles"
]

# Ensure numeric and fill NaNs with 0
df_all[vehicle_cols] = df_all[vehicle_cols].apply(pd.to_numeric, errors="coerce").fillna(0)

# ========== (A) Total vehicles involved across all years ==========
totals = df_all[vehicle_cols].sum().sort_values(ascending=False)

figA, axA = plt.subplots(figsize=(9, 5))
totals.plot(kind="bar", ax=axA)  # many categories -> let matplotlib pick colors (tab10/tab20)
axA.set(
    title="Total Vehicles Involved (All Years)",
    xlabel="Vehicle Type",
    ylabel="Count",
)
axA.bar_label(axA.containers[0], fmt="%.0f", rotation=90, padding=3, fontsize=8)
axA.tick_params(axis="x", labelrotation=30)
for label in axA.get_xticklabels():
    label.set_ha("right")
figA.tight_layout()
figA.savefig(os.path.join(OUTPUT_DIR, "vehicles_involved_total.png"), bbox_inches="tight")

# ========== (B) Stacked by-year counts with % labels on segments ==========
# Aggregate by year
veh_by_year = df_all.groupby("Year")[vehicle_cols].sum().sort_index()

# Percent breakdown per year (for labels only)
veh_pct_by_year = veh_by_year.div(veh_by_year.sum(axis=1), axis=0) * 100

figB, axB = plt.subplots(figsize=(10, 6))
veh_by_year.plot(kind="bar", stacked=True, ax=axB, width=0.85, colormap="tab20")
axB.set(
    title="Vehicles Involved by Year (Stacked)",
    xlabel="Year",
    ylabel="Count",
)
axB.legend(title="Vehicle Type", ncols=2, fontsize=8)

# Add % labels centered on each stacked segment (y-axis remains counts)
# axB.containers order matches columns in veh_by_year
for i, container in enumerate(axB.containers):
    col_name = veh_by_year.columns[i]
    for j, bar in enumerate(container):
        h = bar.get_height()
        if h > 0 and np.isfinite(h):
            pct = veh_pct_by_year.iloc[j, i]
            # Put label in the middle of the segment
            axB.text(
                bar.get_x() + bar.get_width()/2,
                bar.get_y() + h/2,
                f"{pct:.0f}%",
                horizontalalignment="center", verticalalignment="center", fontsize=8, color="white"
            )

figB.tight_layout()
figB.savefig(os.path.join(OUTPUT_DIR, "vehicles_involved_by_year_stacked.png"), bbox_inches="tight")

plt.show()


## Categorical association heatmap

# Temporal

### 1. Yearly injury & fatality overview

In [ ]:
yearly_stats = (
    df_all.groupby("Year")[["MinorInjuries", "SeriousInjuries", "Fatalities"]]
        .sum()
        .sort_index()
)
yearly_stats["Casualties"] = yearly_stats.sum(axis=1)

# Colour-blind-safe palette (orange / blue / green)
cb_colors = ["#E69F00", "#56B4E9", "#009E73"]

# --- Grouped bar chart ---
fig1, ax1 = plt.subplots(figsize=(6, 6))
yearly_stats[["MinorInjuries", "SeriousInjuries", "Fatalities"]].plot(
    kind="bar",
    ax=ax1,
    width=0.8,
    color=cb_colors,
)
ax1.set(
    title="Minor Injuries, Serious Injuries, and Fatalities by Year",
    xlabel="Year",
    ylabel="Count",
)
ax1.legend(title="Type")
fig1.tight_layout()
fig1.savefig(os.path.join(OUTPUT_DIR, "injuries_fatalities_by_year.png"), bbox_inches="tight")

# --- Stacked bar chart (Fatalities at base) ---
fig2, ax2 = plt.subplots(figsize=(6, 6))
yearly_stats[["Fatalities", "SeriousInjuries", "MinorInjuries"]].plot(
    kind="bar",
    ax=ax2,
    stacked=True,
    width=0.8,
    color=cb_colors,  # order matches columns
)
ax2.set(
    title="Minor Injuries, Serious Injuries, and Fatalities (Stacked) by Year",
    xlabel="Year",
    ylabel="Count",
)
ax2.legend(title="Type")
fig2.tight_layout()
fig2.savefig(os.path.join(OUTPUT_DIR, "injuries_fatalities_stacked_by_year.png"), bbox_inches="tight")

plt.show()


In [ ]:
# Compute per-year percentages for labeling
year_totals = yearly_stats["Casualties"].replace(0, pd.NA)
yearly_pct = (
    yearly_stats[["MinorInjuries", "SeriousInjuries", "Fatalities"]]
    .div(year_totals, axis=0) * 100
)

# --- Grouped bar chart with % labels on each bar (y-axis still counts) ---
fig1, ax1 = plt.subplots(figsize=(6, 6))
yearly_stats[["MinorInjuries", "SeriousInjuries", "Fatalities"]].plot(
    kind="bar", ax=ax1, width=0.8, color=cb_colors
)
ax1.set(
    title="Minor Injuries, Serious Injuries, and Fatalities by Year",
    xlabel="Year",
    ylabel="Count",
)
ax1.legend(title="Type")

# Add % labels above each bar
# ax1.containers is ordered per severity series in the same order as plotted
for i, container in enumerate(ax1.containers):
    # i: 0=Minor, 1=Serious, 2=Fatalities
    severity_name = ["MinorInjuries", "SeriousInjuries", "Fatalities"][i]
    for j, bar in enumerate(container):
        h = bar.get_height()
        if h > 0 and pd.notna(yearly_pct.iloc[j][severity_name]):
            pct = yearly_pct.iloc[j][severity_name]
            ax1.text(
                bar.get_x() + bar.get_width()/2,
                h,
                f"{pct:.1f}%",
                ha="center", va="bottom", fontsize=8, rotation=0
            )

fig1.tight_layout()
fig1.savefig(os.path.join(OUTPUT_DIR, "injuries_fatalities_by_year.png"), bbox_inches="tight")

# --- Stacked bar chart (Fatalities at base) with % labels on each segment ---
fig2, ax2 = plt.subplots(figsize=(6, 6))
# Note: columns order must match cb_colors
stack_cols = ["Fatalities", "SeriousInjuries", "MinorInjuries"]
yearly_stats[stack_cols].plot(
    kind="bar", ax=ax2, stacked=True, width=0.8, color=cb_colors
)
ax2.set(
    title="Minor Injuries, Serious Injuries, and Fatalities (Stacked) by Year",
    xlabel="Year",
    ylabel="Count",
)
ax2.legend(title="Type")

# Add % labels centered on each stacked segment
# ax2.containers follows the plotted series order (Fatalities, Serious, Minor)
for i, container in enumerate(ax2.containers):
    # Map i to the correct column name used for % lookup in yearly_pct
    severity_name = ["Fatalities", "SeriousInjuries", "MinorInjuries"][i]
    for j, bar in enumerate(container):
        h = bar.get_height()
        if h > 0 and pd.notna(yearly_pct.iloc[j][severity_name]):
            pct = yearly_pct.iloc[j][severity_name]
            ax2.text(
                bar.get_x() + bar.get_width()/2,
                bar.get_y() + h/2,
                f"{pct:.1f}%",
                ha="center", va="center", fontsize=8, color="white"
            )

fig2.tight_layout()
fig2.savefig(os.path.join(OUTPUT_DIR, "injuries_fatalities_stacked_by_year.png"), bbox_inches="tight")

plt.show()


### 2. Monthly totals (all years combined)


In [ ]:
import calendar

cb_colors = ["#E69F00", "#56B4E9", "#009E73"]

# Ensure `AccidentDate` is datetime and extract month number (1-12)
df_all["AccidentDate"] = pd.to_datetime(df_all["AccidentDate"], errors="coerce")
df_all["Month"] = df_all["AccidentDate"].dt.month

# Aggregate counts per month
monthly_stats = (
    df_all.groupby("Month")[["Fatalities", "SeriousInjuries", "MinorInjuries"]]
        .sum()
        .reindex(range(1, 13))  # guarantees Jan-Dec order even if some months are empty
)
monthly_stats.index = monthly_stats.index.map(lambda m: calendar.month_abbr[m])

# Compute percentages per month (for labels only)
monthly_stats_pct = monthly_stats.div(monthly_stats.sum(axis=1), axis=0) * 100

# Plot stacked bar chart (counts)
fig3, ax3 = plt.subplots(figsize=(10, 6))
monthly_stats.plot(kind="bar", stacked=True, ax=ax3, color=cb_colors)

# Add % labels on each stacked section
for i, container in enumerate(ax3.containers):
    # Each container corresponds to one severity type
    for j, bar in enumerate(container):
        height = bar.get_height()
        if height > 0:  # avoid empty bars
            percent = monthly_stats_pct.iloc[j, i]
            ax3.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_y() + height / 2,
                f"{percent:.1f}%",
                ha="center", va="center", fontsize=8, color="white"
            )

# Formatting
ax3.set(
    title="Injuries and Fatalities by Month (All Years)",
    xlabel="Month",
    ylabel="Count",
)
ax3.legend(title="Severity Level")
fig3.tight_layout()
fig3.savefig(os.path.join(OUTPUT_DIR, "injuries_fatalities_by_month_with_pct.png"), bbox_inches="tight")

plt.show()


### 3. Monthly casualties by year

In [ ]:
monthly_all = (
    df_all.groupby(["Year", "Month"])[["Fatalities", "SeriousInjuries", "MinorInjuries"]]
      .sum()
      .reset_index()
)

# Drop rows where Month is NaN (can happen if AccidentDate was NaT)
monthly_all = monthly_all[monthly_all["Month"].notna()].copy()

# Convert month to integer explicitly (prevents float‑indexing error)
monthly_all["Month"] = monthly_all["Month"].astype(int)

monthly_all["Casualties"] = (
    monthly_all["Fatalities"] + monthly_all["SeriousInjuries"] + monthly_all["MinorInjuries"]
)

# Map month number -> abbreviated name
month_name_map = {i: calendar.month_abbr[i] for i in range(1, 13)}
monthly_all["MonthName"] = monthly_all["Month"].map(month_name_map)

pivot = (
    monthly_all.pivot(index="MonthName", columns="Year", values="Casualties")
      .reindex(["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
)

fig4, ax4 = plt.subplots(figsize=(14, 7))
pivot.plot(kind="bar", ax=ax4, width=0.8)
ax4.set(
    title="Monthly Casualties by Year",
    xlabel="Month",
    ylabel="Casualties",
)
ax4.legend(title="Year")
fig4.tight_layout()
fig4.savefig(os.path.join(OUTPUT_DIR, "monthly_casualties_by_year.png"), bbox_inches="tight")

plt.show()

In [ ]:
# Use the already prepared monthly_all DataFrame which has MonthName and Year

# Create a pivot table where the index is MonthName and the columns are a MultiIndex (injury type and Year)
monthly_pivot = (
    monthly_all.set_index(["MonthName", "Year"])[["Fatalities", "SeriousInjuries", "MinorInjuries"]]
               .unstack("Year")
)

# Reorder the months on the x-axis
months_order = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
monthly_pivot = monthly_pivot.reindex(months_order)

fig4, ax4 = plt.subplots(figsize=(14, 7))
monthly_pivot.plot(
    kind="bar",
    stacked=True,
    ax=ax4,
    width=0.8
)
ax4.set(
    title="Monthly Injuries Breakdown by Year",
    xlabel="Month",
    ylabel="Injury Count"
)
ax4.legend(title="Year", bbox_to_anchor=(1.05, 1), loc='upper left')
fig4.tight_layout()
fig4.savefig(os.path.join(OUTPUT_DIR, "monthly_injuries_stacked_by_year.png"), bbox_inches="tight")

plt.show()


## Location

In [ ]:
def get_basemap_provider():
    """Return a reliable XYZ tile provider object for contextily.

    Iterates through a short list of well‑known providers and returns the
    **first** one that is available (i.e. contains a valid tile `url`).
    """
    candidate_paths = [
        ("OpenStreetMap", "Mapnik"),
        ("CartoDB", "Positron"),
        ("Stamen", "TonerLite"),
        ("Stamen", "Toner"),
    ]

    for path in candidate_paths:
        try:
            provider = ctx.providers
            for part in path:  # walk the nested dict/namespace
                provider = provider[part]
            if "url" in provider:  # basic sanity check
                return provider
        except (KeyError, AttributeError, TypeError):
            continue  # try the next one

    raise RuntimeError("No valid XYZ tile provider found in ctx.providers")


def savefig(fig: plt.Figure, filename: str):
    if isinstance(filename, str):
        filename = Path(filename)  # ensure Path object
    fig.savefig(OUTPUT_DIR / filename, bbox_inches="tight")

basemap_source = get_basemap_provider()

In [ ]:
# Load OSM boundary data using OSMnx
import osmnx as ox
# Define the area of interest
area_name = "Thailand"
# Download the boundary polygon for Bangkok
Thailand_boundary = ox.geocode_to_gdf(area_name)
# Ensure the geometry is in the correct CRS (WGS84)
Thailand_boundary = Thailand_boundary.to_crs(epsg=4326)
# Save the boundary to a GeoJSON file
output_path = os.path.join("osm", "Thailand_boundary.geojson")
Thailand_boundary.to_file(output_path, driver='GeoJSON')

Bangkok_boundary = ox.geocode_to_gdf("Bangkok, Thailand")
Bangkok_boundary = Bangkok_boundary.to_crs(epsg=4326)
output_path_bangkok = os.path.join("osm", "Bangkok_boundary.geojson")
Bangkok_boundary.to_file(output_path_bangkok, driver='GeoJSON')

In [ ]:
def get_basemap_provider():
    """Return a reliable XYZ tile provider object for contextily.

    Iterates through a short list of well‑known providers and returns the
    **first** one that is available (i.e. contains a valid tile `url`).
    """
    candidate_paths = [
        ("OpenStreetMap", "Mapnik"),
        ("CartoDB", "Positron"),
        ("Stamen", "TonerLite"),
        ("Stamen", "Toner"),
    ]

    for path in candidate_paths:
        try:
            provider = ctx.providers
            for part in path:  # walk the nested dict/namespace
                provider = provider[part]
            if "url" in provider:  # basic sanity check
                return provider
        except (KeyError, AttributeError, TypeError):
            continue  # try the next one

    raise RuntimeError("No valid XYZ tile provider found in ctx.providers")


def savefig(fig: plt.Figure, filename: str):
    if isinstance(filename, str):
        filename = Path(filename)  # ensure Path object
    fig.savefig(OUTPUT_DIR / filename, bbox_inches="tight")

basemap_source = get_basemap_provider()

# load osm (boundaries) data using osmnx
import osmnx as ox
# Define the area of interest
area_name = "Thailand"
# Download the boundary polygon for Thailand
Thailand_boundary = ox.geocode_to_gdf(area_name)
Thailand_boundary = Thailand_boundary.to_crs(epsg=4326)
output_path = os.path.join("osm", "Thailand_boundary.geojson")
Thailand_boundary.to_file(output_path, driver='GeoJSON')

Bangkok_boundary = ox.geocode_to_gdf("Bangkok, Thailand")
Bangkok_boundary = Bangkok_boundary.to_crs(epsg=4326)
output_path_bangkok = os.path.join("osm", "Bangkok_boundary.geojson")
Bangkok_boundary.to_file(output_path_bangkok, driver='GeoJSON')

# --- Clean & prep coordinates -------------------------------------------------
coords = df_all.dropna(subset=["Latitude", "Longitude"]).copy()
coords[["Latitude", "Longitude"]] = coords[["Latitude", "Longitude"]].apply(pd.to_numeric, errors="coerce")
coords = coords.dropna(subset=["Latitude", "Longitude"])

# GeoDataFrame in WGS84
points_wgs = gpd.GeoDataFrame(coords, geometry=gpd.points_from_xy(coords.Longitude, coords.Latitude), crs="EPSG:4326")

# Restrict to Thailand
points_wgs = gpd.clip(points_wgs, Thailand_boundary)

# ------------------------ severity classification ----------------------------
conditions = [
    points_wgs["Fatalities"] > 0,
    points_wgs["SeriousInjuries"] > 0,
]
choices = ["Fatal", "Serious"]
points_wgs["Severity"] = np.select(conditions, choices, default="Minor")
color_map = {"Fatal": "red", "Serious": "orange", "Minor": "blue"}

# Reproject for web tiles
points_web = points_wgs.to_crs(epsg=3857)
thai_boundary_web = Thailand_boundary.to_crs(epsg=3857)

# ------------------------ map 4a: all points ---------------------------------
fig_t, ax_t = plt.subplots(figsize=(10, 10))
thai_boundary_web.boundary.plot(ax=ax_t, edgecolor="grey", linewidth=2)
points_web.plot(ax=ax_t, color="steelblue", markersize=3, alpha=0.4)
ctx.add_basemap(ax_t, source=basemap_source, attribution_size=6, reset_extent=False)
ax_t.set_title("TRAMS Accident Locations (2019-2024)")
ax_t.set_axis_off()
ax_t.set_xlim(thai_boundary_web.total_bounds[[0, 2]])
ax_t.set_ylim(thai_boundary_web.total_bounds[[1, 3]])
fig_t.tight_layout()
savefig(fig_t, "accident_locations_thailand.png")

# ------------------------ map 4b: severity‑coloured ---------------------------
fig_s, ax_s = plt.subplots(figsize=(10, 10))
for sev, col in color_map.items():
    points_web.query("Severity == @sev").plot(ax=ax_s, color=col, markersize=.1, alpha=0.5, label=sev)
thai_boundary_web.boundary.plot(ax=ax_s, edgecolor="grey", linewidth=2)
ctx.add_basemap(ax_s, source=basemap_source, attribution_size=6, reset_extent=False)
ax_s.set_title("Accident Locations by Severity (2019-2024)")
ax_s.set_axis_off()
ax_s.set_xlim(thai_boundary_web.total_bounds[[0, 2]])
ax_s.set_ylim(thai_boundary_web.total_bounds[[1, 3]])
ax_s.legend()
fig_s.tight_layout()
savefig(fig_s, "accident_locations_severity.png")

In [ ]:
df_all

In [ ]:
# ------------------------ map 4a: all points ---------------------------------
fig_t, ax_t = plt.subplots(figsize=(10, 10))
thai_boundary_web.boundary.plot(ax=ax_t, edgecolor="grey", linewidth=2)
points_web.plot(ax=ax_t, color="steelblue", markersize=3, alpha=0.4)
ctx.add_basemap(ax_t, source=basemap_source, attribution_size=6, reset_extent=False)
ax_t.set_title("TRAMS Accident Locations (2019-2024)")
ax_t.set_axis_off()
ax_t.set_xlim(thai_boundary_web.total_bounds[[0, 2]])
ax_t.set_ylim(thai_boundary_web.total_bounds[[1, 3]])
fig_t.tight_layout()
savefig(fig_t, "accident_locations_thailand.png")

# Ensure color_map is a dict, not a numpy array
color_map = {"Fatal": "red", "Serious": "orange", "Minor": "blue"}

# ------------------------ map 4b: severity‑coloured ---------------------------
fig_s, ax_s = plt.subplots(figsize=(10, 10))
for sev, col in color_map.items():
    points_web.query("Severity == @sev").plot(ax=ax_s, color=col, markersize=.1, alpha=0.5, label=sev)
thai_boundary_web.boundary.plot(ax=ax_s, edgecolor="grey", linewidth=2)
ctx.add_basemap(ax_s, source=basemap_source, attribution_size=6, reset_extent=False)
ax_s.set_title("Accident Locations by Severity (2019-2024)")
ax_s.set_axis_off()
ax_s.set_xlim(thai_boundary_web.total_bounds[[0, 2]])
ax_s.set_ylim(thai_boundary_web.total_bounds[[1, 3]])
ax_s.legend()
fig_s.tight_layout()
savefig(fig_s, "accident_locations_severity.png")

# --- Separate maps for each severity level ---
for sev, col in color_map.items():
    fig_sev, ax_sev = plt.subplots(figsize=(10, 10))
    points_web.query("Severity == @sev").plot(ax=ax_sev, color=col, markersize=.1, alpha=0.5, label=sev)
    thai_boundary_web.boundary.plot(ax=ax_sev, edgecolor="grey", linewidth=2)
    ctx.add_basemap(ax_sev, source=basemap_source, attribution_size=6, reset_extent=False)
    ax_sev.set_title(f"Accident Locations: {sev} (2019-2024)")
    ax_sev.set_axis_off()
    ax_sev.set_xlim(thai_boundary_web.total_bounds[[0, 2]])
    ax_sev.set_ylim(thai_boundary_web.total_bounds[[1, 3]])
    ax_sev.legend()
    fig_sev.tight_layout()
    savefig(fig_sev, f"accident_locations_{sev.lower()}.png")

# ------------------------ map 4c: Bangkok zoom -------------------------------
if "Bangkok_boundary" in globals():
    bkk_boundary_wgs = Bangkok_boundary.to_crs("EPSG:4326")
else:
    bkk_bbox = box(100.3, 13.4, 100.9, 13.95)
    bkk_boundary_wgs = gpd.GeoDataFrame({"geometry": [bkk_bbox]}, crs="EPSG:4326")

points_bkk_web = points_wgs.to_crs(epsg=3857)
bkk_boundary_web = bkk_boundary_wgs.to_crs(epsg=3857)

fig_b, ax_b = plt.subplots(figsize=(10, 10))
bkk_boundary_web.boundary.plot(ax=ax_b, edgecolor="red", linewidth=2)
for sev, col in color_map.items():
    points_bkk_web.query("Severity == @sev").plot(ax=ax_b, color=col, markersize=3, alpha=0.5, label=sev)
ctx.add_basemap(ax_b, source=basemap_source, attribution_size=6, reset_extent=False)
ax_b.set_title("Accident Locations - Bangkok")
ax_b.set_axis_off()
ax_b.set_xlim(bkk_boundary_web.total_bounds[[0, 2]])
ax_b.set_ylim(bkk_boundary_web.total_bounds[[1, 3]])
ax_b.legend()
fig_b.tight_layout()
savefig(fig_b, "accident_locations_bangkok.png")

plt.show()

# Ensure color_map is a dict, not a numpy array
color_map = {"Fatal": "red", "Serious": "orange", "Minor": "blue"}

# ------------------------ map 4b: severity‑coloured ---------------------------
fig_s, ax_s = plt.subplots(figsize=(10, 10))
for sev, col in color_map.items():
    points_web.query("Severity == @sev").plot(ax=ax_s, color=col, markersize=.1, alpha=0.5, label=sev)
thai_boundary_web.boundary.plot(ax=ax_s, edgecolor="grey", linewidth=2)
ctx.add_basemap(ax_s, source=basemap_source, attribution_size=6, reset_extent=False)
ax_s.set_title("Accident Locations by Severity (2019-2024)")
ax_s.set_axis_off()
ax_s.set_xlim(thai_boundary_web.total_bounds[[0, 2]])
ax_s.set_ylim(thai_boundary_web.total_bounds[[1, 3]])
ax_s.legend()
fig_s.tight_layout()
savefig(fig_s, "accident_locations_severity.png")

# --- Separate maps for each severity level ---
for sev, col in color_map.items():
    fig_sev, ax_sev = plt.subplots(figsize=(10, 10))
    points_web.query("Severity == @sev").plot(ax=ax_sev, color=col, markersize=.1, alpha=0.5, label=sev)
    thai_boundary_web.boundary.plot(ax=ax_sev, edgecolor="grey", linewidth=2)
    ctx.add_basemap(ax_sev, source=basemap_source, attribution_size=6, reset_extent=False)
    ax_sev.set_title(f"Accident Locations: {sev} (2019-2024)")
    ax_sev.set_axis_off()
    ax_sev.set_xlim(thai_boundary_web.total_bounds[[0, 2]])
    ax_sev.set_ylim(thai_boundary_web.total_bounds[[1, 3]])
    ax_sev.legend()
    fig_sev.tight_layout()
    savefig(fig_sev, f"accident_locations_{sev.lower()}.png")

# ------------------------ map 4c: Bangkok zoom -------------------------------
if "Bangkok_boundary" in globals():
    bkk_boundary_wgs = Bangkok_boundary.to_crs("EPSG:4326")
else:
    bkk_bbox = box(100.3, 13.4, 100.9, 13.95)
    bkk_boundary_wgs = gpd.GeoDataFrame({"geometry": [bkk_bbox]}, crs="EPSG:4326")

points_bkk_web = points_wgs.to_crs(epsg=3857)
bkk_boundary_web = bkk_boundary_wgs.to_crs(epsg=3857)

fig_b, ax_b = plt.subplots(figsize=(10, 10))
bkk_boundary_web.boundary.plot(ax=ax_b, edgecolor="red", linewidth=2)
for sev, col in color_map.items():
    points_bkk_web.query("Severity == @sev").plot(ax=ax_b, color=col, markersize=3, alpha=0.5, label=sev)
ctx.add_basemap(ax_b, source=basemap_source, attribution_size=6, reset_extent=False)
ax_b.set_title("Accident Locations - Bangkok")
ax_b.set_axis_off()
ax_b.set_xlim(bkk_boundary_web.total_bounds[[0, 2]])
ax_b.set_ylim(bkk_boundary_web.total_bounds[[1, 3]])
ax_b.legend()
fig_b.tight_layout()
savefig(fig_b, "accident_locations_bangkok.png")

plt.show()

In [ ]:
import os
import calendar
from pathlib import Path

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
from shapely.geometry import box
import numpy as np

# -------------------------------------------------------------
# Defaults & output folder
# -------------------------------------------------------------
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["savefig.dpi"] = 300

OUTPUT_DIR = Path("../data/DOH")
OUTPUT_DIR.mkdir(exist_ok=True)

# `df_all` and `Thailand_boundary` are assumed to be pre‑loaded in memory. If not,
# read them here (paths omitted for brevity).

# %% [markdown]
# ## Helper utilities

# %%

def get_basemap_provider():
    """Return a reliable XYZ tile provider object for contextily.

    Iterates through a short list of well‑known providers and returns the
    **first** one that is available (i.e. contains a valid tile `url`).
    """
    candidate_paths = [
        ("OpenStreetMap", "Mapnik"),
        ("CartoDB", "Positron"),
        ("Stamen", "TonerLite"),
        ("Stamen", "Toner"),
    ]

    for path in candidate_paths:
        try:
            provider = ctx.providers
            for part in path:  # walk the nested dict/namespace
                provider = provider[part]
            if "url" in provider:  # basic sanity check
                return provider
        except (KeyError, AttributeError, TypeError):
            continue  # try the next one

    raise RuntimeError("No valid XYZ tile provider found in ctx.providers")


def savefig(fig: plt.Figure, filename: str):
    if isinstance(filename, str):
        filename = Path(filename)  # ensure Path object
    fig.savefig(OUTPUT_DIR / filename, bbox_inches="tight")
    fig.savefig(OUTPUT_DIR / filename, bbox_inches="tight")


basemap_source = get_basemap_provider()

# ## 1. Yearly injury & fatality overview

# %%
yearly_stats = (
    df_all.groupby("Year")[["MinorInjuries", "SeriousInjuries", "Fatalities"]]
      .sum()
      .sort_index()
)
yearly_stats["Casualties"] = yearly_stats.sum(axis=1)

cb_colors = ["#E69F00", "#56B4E9", "#009E73"]

fig1, ax1 = plt.subplots(figsize=(6, 6))
yearly_stats[["MinorInjuries", "SeriousInjuries", "Fatalities"]].plot(
    kind="bar", ax=ax1, width=0.8, color=cb_colors
)
ax1.set(title="Minor, Serious & Fatal Injuries by Year", xlabel="Year", ylabel="Count")
ax1.legend(title="Type")
fig1.tight_layout(); savefig(fig1, "injuries_fatalities_by_year.png")

fig2, ax2 = plt.subplots(figsize=(6, 6))
yearly_stats[["Fatalities", "SeriousInjuries", "MinorInjuries"]].plot(
    kind="bar", ax=ax2, stacked=True, width=0.8, color=cb_colors
)
ax2.set(title="Minor, Serious & Fatal Injuries (Stacked)", xlabel="Year", ylabel="Count")
ax2.legend(title="Type")
fig2.tight_layout(); savefig(fig2, "injuries_fatalities_stacked_by_year.png")

plt.show()

# ## 2. Monthly totals (all years combined)

# %%
df_all["AccidentDate"] = pd.to_datetime(df_all["AccidentDate"], errors="coerce")
df_all["Month"] = df_all["AccidentDate"].dt.month

monthly_stats = (
    df_all.groupby("Month")[["Fatalities", "SeriousInjuries", "MinorInjuries"]]
      .sum()
      .reindex(range(1, 13))
)
monthly_stats.index = monthly_stats.index.map(lambda m: calendar.month_abbr[m])

fig3, ax3 = plt.subplots(figsize=(10, 6))
monthly_stats.plot(kind="bar", ax=ax3)
ax3.set(title="Injuries & Fatalities by Month (All Years)", xlabel="Month", ylabel="Count")
ax3.legend(title="Type")
fig3.tight_layout(); savefig(fig3, "injuries_fatalities_by_month.png")

plt.show()

# ## 3. Monthly casualties by year

# %%
monthly_all = (
    df_all.groupby(["Year", "Month"])[["Fatalities", "SeriousInjuries", "MinorInjuries"]]
      .sum()
      .reset_index()
)
monthly_all = monthly_all.dropna(subset=["Month"]).copy()
monthly_all["Month"] = monthly_all["Month"].astype(int)
monthly_all["Casualties"] = monthly_all[["Fatalities", "SeriousInjuries", "MinorInjuries"]].sum(axis=1)
month_map = {i: calendar.month_abbr[i] for i in range(1, 13)}
monthly_all["MonthName"] = monthly_all["Month"].map(month_map)

pivot = (
    monthly_all.pivot(index="MonthName", columns="Year", values="Casualties")
      .reindex(["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
)

fig4, ax4 = plt.subplots(figsize=(14, 7))
pivot.plot(kind="bar", ax=ax4, width=0.8)
ax4.set(title="Monthly Casualties by Year", xlabel="Month", ylabel="Casualties")
ax4.legend(title="Year")
fig4.tight_layout(); savefig(fig4, "monthly_casualties_by_year.png")

plt.show()



In [ ]:
# %% [markdown]
# ## 4. Accident‑location maps (Thailand & Bangkok)

# --- Clean & prep coordinates -------------------------------------------------
coords = df_all.dropna(subset=["Latitude", "Longitude"]).copy()
coords[["Latitude", "Longitude"]] = coords[["Latitude", "Longitude"]].apply(pd.to_numeric, errors="coerce")
coords = coords.dropna(subset=["Latitude", "Longitude"])

points_wgs = gpd.GeoDataFrame(
    coords,
    geometry=gpd.points_from_xy(coords.Longitude, coords.Latitude),
    crs="EPSG:4326",
)

# Restrict to Thailand (keeps compute light, but we'll still plot full boundary)
points_wgs = gpd.clip(points_wgs, Thailand_boundary)

# --------------------- severity classification ----------------------------
conditions = [points_wgs["Fatalities"] > 0, points_wgs["SeriousInjuries"] > 0]
choices    = ["Fatal", "Serious"]
points_wgs["Severity"] = np.select(conditions, choices, default="Minor")
color_map = {"Fatal": "red", "Serious": "orange", "Minor": "blue"}

# --------------------- project once for all plots ------------------------
points_web        = points_wgs.to_crs(epsg=3857)
thai_boundary_web = Thailand_boundary.to_crs(epsg=3857)

# Helper -------------------------------------------------------------------

def plot_map(points, boundary, title, fname, extent=None, colour_by=None, zoom=None):
    fig, ax = plt.subplots(figsize=(10, 10))
    boundary.boundary.plot(ax=ax, edgecolor="grey", linewidth=2)

    if colour_by is None:
        points.plot(ax=ax, color="steelblue", markersize=3, alpha=0.4)
    else:
        for cat, col in colour_by.items():
            points.query("Severity == @cat").plot(
                ax=ax, color=col, markersize=3, alpha=0.5, label=cat
            )

    ctx.add_basemap(ax, source=basemap_source, zoom=zoom, attribution_size=6, reset_extent=False)

    if extent is None:
        extent = boundary.total_bounds  # [minx, miny, maxx, maxy]
    ax.set_xlim(extent[[0, 2]]); ax.set_ylim(extent[[1, 3]])

    ax.set_title(title)
    ax.set_axis_off()
    if colour_by is not None:
        ax.legend()
    fig.tight_layout()
    savefig(fig, fname)

### 4.1  National‑scale maps

# 4.1a - All accidents
plot_map(
    points_web,
    thai_boundary_web,
    title="TRAMS Accident Locations (2019-2024)",
    fname="accident_locations_thailand.png",
    zoom=5,
)

# 4.1b - Severity‑coloured
plot_map(
    points_web,
    thai_boundary_web,
    title="Accident Locations by Severity (2019-2024)",
    fname="accident_locations_severity.png",
    colour_by=color_map,
    zoom=5,
)

# 4.1c - Separate layer per severity (optional heavy render)
for sev, col in color_map.items():
    sub = points_web.query("Severity == @sev")
    if sub.empty:
        continue
    plot_map(
        sub,
        thai_boundary_web,
        title=f"Accident Locations: {sev} (2019-2024)",
        fname=f"accident_locations_{sev.lower()}.png",
        zoom=5,
    )

### 4.2  Bangkok zoom (no point‑clipping)

# Boundary or fallback bbox
if "Bangkok_boundary" in globals():
    bkk_boundary_wgs = Bangkok_boundary.to_crs("EPSG:4326")
else:
    bkk_bbox = box(100.3, 13.4, 100.9, 13.95)
    bkk_boundary_wgs = gpd.GeoDataFrame({"geometry": [bkk_bbox]}, crs="EPSG:4326")

bkk_boundary_web = bkk_boundary_wgs.to_crs(epsg=3857)

# Use *all* accidents; just set the extent to Bangkok bounds for a clean zoom
plot_map(
    points_web,
    bkk_boundary_web,
    title="Accident Locations - Bangkok",
    fname="accident_locations_bangkok.png",
    extent=bkk_boundary_web.total_bounds,
    colour_by=color_map,
    zoom=11,          # high‑detail tiles for city view
)

### All done
# Every figure is written to the `plots/` directory with axes tightly fitted to their respective boundaries and a high‑resolution basemap for the Bangkok zoom.  
# Every figure has been saved to the `plots/` directory.


## Location